# 11. Counting crossings in a reticle

**The question.** A sensor makes far more hits than the detector can send out, so the study asks
whether logic at the edge of the sensor could replace the hit map of each reticle by one number,
the count of crossings in it. That logic sees only the bits of its own reticle: no charge, no
neighbouring reticles, no position of the shower axis. A reticle is a block of 21 x 13 pixels.
The detector reads it as a hit map: one bit per pixel, on or off. Given only that map, how well
can we estimate the number of crossings in the reticle, and why do the simple ways fail where
they fail?

This notebook builds four estimators, from the simplest to the richest, and scores each on showers
it never saw:

1. count the hit pixels and convert with a table;
2. count the clusters (sets of touching hit pixels) and convert with a table;
3. look up every cluster's shape in a dictionary of "crossings per shape" and add up;
4. boosted decision trees on 49 numbers that summarise the hit map.

Each table is first built with plain numpy and checked against the study code
(`analysis/pixelav/mip_counting.py`), which is then used for the scoring.

**Where this notebook sits in the chain**

```
Geant4 shower: the energy deposited in every step in the silicon
  -> PIXELAV: the charge each particle leaves on each pixel       charges_b*.npz
  -> reticles.py: blocks of 21 x 13 pixels and their truth        ret_K1.npz, ret_K8.npz  (200 showers)
  -> notebook 10: one plane, its reticles, rising density
  -> this notebook: the count of each reticle estimated from its hit map
  -> notebook 12: the counts summed over a shower
```

**What you need beforehand**

- Notebook 10. It defines the crossing (the truth object), the reticle, the readout and the
  truth `n_frac`, and it measures how hit pixels and clusters behave as the density rises. This
  notebook uses those words without defining them again.
- Two input files, in `models/reticle_study/` or in the directory named by the environment
  variable `CALOMAPS_RETICLE_STUDY`:
  - `ret_K1.npz`: 200 showers of 50 GeV photons, cut into reticles (only the reticles that hold
    charge are stored);
  - `ret_K8.npz`: the same 200 showers stacked 8 at a time, plane by plane, to raise the density.

  Both are written by `analysis/pixelav/reticles.py` from the charge maps of
  `analysis/pixelav/charge_maps.py`, which run PIXELAV on the Geant4 steps. Making them again needs
  PIXELAV and a sensor field map that is not part of this repository, so this notebook is kept
  with its outputs. The commands are in section 9 of `docs/reticle_mip_counting.md`.
- The study behind this notebook is `docs/reticle_mip_counting.md`; its sections 5 and 6 report
  the results that this notebook rebuilds step by step.

**Words used below**

- *estimator*: a rule that turns a hit map into a number, the estimated count.
- *truth*: `n_frac`, the number of crossings in the reticle, where a crossing that spreads over
  several reticles is shared among them in proportion to its charge in each (notebook 10,
  section 6). It is fractional. A crossing counts when its charge on the plane is above the
  threshold.
- *bias*: the mean of (estimate - truth) over a set of reticles. *rms*: the square root of the
  mean of (estimate - truth) squared; it includes the bias.
- *training* and *test* showers: tables and models are fitted on the training showers only, and
  every score is computed on the test showers.

## 1. Loading, readout, truth and the split

`mc.prepare` does in one call what notebook 10 did by hand: it reads the hit map of every reticle
out of its charge image (Gaussian noise added to every pixel, then the threshold), computes the
truths, finds the clusters and the 49 summary numbers of section 6, and assigns each shower to one
of five blocks of 40 showers, the blocks used by the write-up. We call it on both files at once.

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap
from scipy import ndimage
import sklearn

T_START = time.time()
REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA = os.environ.get('CALOMAPS_RETICLE_STUDY', os.path.join(REPO, 'models', 'reticle_study'))
sys.path.insert(0, os.path.join(REPO, 'analysis', 'pixelav'))
import mip_counting as mc        # the counting study (analysis/pixelav/mip_counting.py)


def need(name):
    # the path of an input file, or a clear error naming it
    path = os.path.join(DATA, name)
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"{path} is missing. The ret_K*.npz files are made by analysis/pixelav/reticles.py; see "
            "docs/reticle_mip_counting.md section 9 for the commands. Put them in models/reticle_study/ "
            "or point CALOMAPS_RETICLE_STUDY at their directory.")
    return path


THRESHOLD = mc.THRESHOLD   # electrons: a pixel above this is hit (on)
NOISE = mc.NOISE           # electrons: Gaussian noise per pixel (sigma)
SEED = mc.SEED             # the noise of the readout is drawn from this seed
N_BLOCKS = mc.N_FOLDS      # the write-up cuts the showers into this many blocks
N_TRAIN = 160              # showers 0 to 159 train every table and model; 160 to 199 test them
M = mc.MIN_COUNT           # training entries a table entry needs before the study code uses it alone
PLANE_MIN = 20             # sensor planes with fewer true crossings are left out of the plane ratios
NAMES = ('hits', 'clusters', 'cluster_shape', 'gbt', 'gbt_layer')
COLOUR = {nm: mc.COLOURS[nm] for nm in NAMES + ('analog',)}
STYLE = {nm: '-' for nm in NAMES}
STYLE.update(gbt_layer='--', analog=':')
print(f"repository: {REPO}\ndata: {DATA}")
print(f"threshold {THRESHOLD:g} e, noise {NOISE:g} e, seed {SEED}; training showers 0 to {N_TRAIN - 1}, "
      f"test showers from {N_TRAIN}; a table entry stands alone from {M} training entries")
print(f"numpy {np.__version__}, scikit-learn {sklearn.__version__}")

In [ ]:
R1 = mc.load_reticles(need('ret_K1.npz'))        # load_reticles applies mc.with_truth_objects
R8 = mc.load_reticles(need('ret_K8.npz'))
with np.load(need('ret_K1.npz')) as d:           # arrays that load_reticles leaves out
    R1['part_p_MeV'] = d['part_p_MeV']           # momentum of the particle behind each row (section 9)
    AXIS_MODULE = int(d['axis_module'])          # the module that the shower axis crosses
    GUN = f"{float(d['sim_gun_energy_GeV']):g} GeV {d['sim_gun_particle']}"
with np.load(need('ret_K8.npz')) as d:
    assert int(d['axis_module']) == AXIS_MODULE, "the two files disagree on the axis module"
t0 = time.time()
(D1, D8), EV_BLOCK = mc.prepare([R1, R8], ['K1', 'K8'], THRESHOLD, NOISE, SEED, N_BLOCKS, keep_hits=True)
del R8['img']                                    # the K8 charge images are not needed further
HITS1 = D1['hits']                               # [reticle, row u, column v], bool
NY, NX = D1['ny'], D1['nx']
PX, PY = D1['pitch_x'], D1['pitch_y']            # um: 50 along v (columns), 25 along u (rows)
EXT = (0, NX * PX, 0, NY * PY)                   # imshow extent in um, so that pixels keep their shape
N_LAYERS = int(D1['layer'].max()) + 1
print(f"showers of {GUN}; mc.prepare took {time.time() - t0:.0f} s; reticles of {NX} x {NY} pixels of "
      f"{PX:g} x {PY:g} um, {N_LAYERS} layers")
for D in (D1, D8):
    what = 'single showers' if D['overlay'] == 1 else f"stacks of {D['overlay']} showers"
    print(f"{D['tag']}: {D['n']:,} reticles holding charge from {D['n_super_events']} {what}; "
          f"{len(D['pix']['idx']):,} hit pixels in {len(D['cl']['size']):,} clusters")
print("\nwhat mc.prepare returns per file (K1 shown):")
for k, v in D1.items():
    if isinstance(v, np.ndarray) and v.ndim >= 1 and len(v) == D1['n']:
        print(f"  per reticle     {k:12s} {str(v.dtype):8s} {v.shape}")
print(f"  per hit pixel   pix: {', '.join(D1['pix'])}")
print(f"  per cluster     cl:  {', '.join(D1['cl'])}")

The arrays used below:

- `n_frac`: the truth (`n_own` and `n_in`, the other two counting rules of notebook 10, are not
  used here);
- `n_hits`, `n_clusters`: hit pixels and clusters in the hit map; `hits`: the hit maps themselves;
- `feat`: the 49 summary numbers of section 6; `layer`, `module`, `r_mm` (distance of the reticle
  centre from the shower axis), `q_sum` (the charge of the reticle before noise);
- `super_evt`: the shower (in K8, the stack); `fold`: its block of 40 showers.

Before using them, three checks of what they hold.

In [ ]:
assert np.array_equal(D1['n_hits'], HITS1.sum(axis=(1, 2))), "n_hits is not the number of hit pixels"
print("check: n_hits is the number of hit pixels of every K1 hit map")

img = R1['img']
rng = np.random.default_rng((SEED, 0))          # mc.prepare draws the noise of file i from (SEED, i); K1 is file 0
same = True
for a in range(0, len(img), mc.CHUNK):
    b = img[a:a + mc.CHUNK]
    on = b + rng.standard_normal(b.shape, dtype=np.float32) * np.float32(NOISE) > THRESHOLD
    same = same and np.array_equal(on, HITS1[a:a + mc.CHUNK])
assert same, "the hit map is not the charge plus the readout's noise above the threshold"
print("check: the readout's noise, redrawn from its seed, gives the same hit map in every K1 reticle")
print(f"hit pixels without any charge (made by noise alone): {int(np.count_nonzero(HITS1 & (img == 0)))}")

info = D1['truth_info']
gap = abs(D1['n_frac'].sum() - info['above_cut'])
assert gap <= 1e-3 + info['above_cut_incomplete'], f"n_frac sums to {D1['n_frac'].sum():.3f}, not to the crossings"
print(f"check: n_frac sums to the {info['above_cut']:,} crossings above {THRESHOLD:g} e of the 200 showers "
      f"(difference {gap:.1e})")

**The split.** Showers 0 to 159 are the training showers, 160 to 199 the test showers. The split
is by shower, not by reticle: neighbouring reticles of one shower are alike, and a table fitted on
some reticles of a shower would partly know its other reticles. The cell below reads from the file
which showers lie behind each reticle (every particle row carries its shower number) and checks
that no shower is on both sides. It also defines `training`, the only way a mask of reticles
reaches the fits of sections 3 to 6, 10 and 11: it refuses any reticle of a test shower. Section
7b refits on the other blocks only to compare with the write-up (there and in section 9), and says
so.

In [ ]:
TRAIN = D1['super_evt'] < N_TRAIN
TEST = ~TRAIN
row_ret = R1['part_reticle']
sh_train = np.unique(R1['part_evt'][TRAIN[row_ret]])
sh_test = np.unique(R1['part_evt'][TEST[row_ret]])
assert np.array_equal(sh_train, np.arange(N_TRAIN)) and \
       np.array_equal(sh_test, np.arange(N_TRAIN, R1['n_events'])), "the split is not showers 0-159 / 160-199"
print(f"check: the training reticles hold particles of showers {sh_train[0]} to {sh_train[-1]} only, the test "
      f"reticles of showers {sh_test[0]} to {sh_test[-1]} only")
mc.check_unseen(R1['n_events'], [D1], [TRAIN], [D1], [TEST], 'the split')    # raises mc.Inconsistent otherwise
print("check: mc.check_unseen finds no shower in both sets")
assert np.array_equal(TEST, D1['fold'] == N_BLOCKS - 1), "the test showers are not the write-up's last block"
print(f"check: the test showers are the last of the write-up's {N_BLOCKS} blocks")


def training(mask):
    # the reticles a table or model may be fitted on: never a reticle of a test shower
    n_bad = int(np.count_nonzero(mask & TEST))
    assert n_bad == 0, f"a training selection holds {n_bad:,} reticles of test showers"
    return mask


VT = mc.view([D1], [TEST])      # the test reticles and their clusters, in the form the estimators read
TRUTH_T = VT['n_frac']
TEST_IDX = np.flatnonzero(TEST)  # row j of VT is reticle TEST_IDX[j] of K1
print(f"\ntraining: {int(TRAIN.sum()):,} reticles, {D1['n_frac'][TRAIN].sum():,.0f} crossings; "
      f"test: {int(TEST.sum()):,} reticles, {TRUTH_T.sum():,.0f} crossings")

**What a hit map looks like.** Six training reticles on the module that the shower axis crosses,
with about 1, 2, 4, 8, 15 and 30 crossings. Each pixel is drawn with its true shape, twice as wide
(v, 50 um) as it is tall (u, 25 um). Look at how the pattern changes from separate spots to
touching patches.

In [ ]:
def draw_hits(ax, hits, title):
    ax.imshow(hits, origin='lower', interpolation='nearest', extent=EXT,
              cmap=ListedColormap(['white', '0.15']), vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('v [um]', fontsize=9)
    ax.set_ylabel('u [um]', fontsize=9)
    ax.tick_params(labelsize=8)


def pick_near(mask, truth, target, rng):
    # a random reticle of mask whose truth is close to target
    c = np.flatnonzero(mask & (np.abs(truth - target) < max(0.05, 0.04 * target)))
    if len(c) == 0:
        c = np.flatnonzero(mask & (np.abs(truth - target) == np.min(np.abs(truth[mask] - target))))
    return int(rng.choice(c))


rng = np.random.default_rng(SEED)
on_axis = D1['module'] == AXIS_MODULE
ex = [pick_near(TRAIN & on_axis, D1['n_frac'], t, rng) for t in (1, 2, 4, 8, 15, 30)]
fig, axs = plt.subplots(3, 2, figsize=(12, 6.2), constrained_layout=True)
for ax, i in zip(axs.flat, ex):
    draw_hits(ax, HITS1[i], f"n_frac = {D1['n_frac'][i]:.2f}: {D1['n_hits'][i]} hit pixels, "
                            f"layer {D1['layer'][i]}, {D1['r_mm'][i]:.1f} mm from the axis")
plt.show()

The count is easy to read off in a sparse map and hard in a dense one, where the spots touch and
the number of hit pixels per crossing changes. The estimators below try to do the reading
automatically.

## 2. How an estimator is scored

Every estimator is scored the same way, on the test showers only:

- **per bin of the true count**: the test reticles are grouped by their truth rounded to an
  integer, in the bins of the study (`mc.TRUTH_BINS`), and in each bin we give the bias and the rms
  of (estimate - truth). The bin "0" holds reticles with less than half a crossing: small shares
  of crossings that lie mostly in a neighbouring reticle.
- **summed over a sensor plane**: the estimates of all reticles of one plane (one shower, one
  module, one layer) are added up and divided by the true crossings of that plane. We give the
  mean and the standard deviation of this ratio over the planes holding at least `PLANE_MIN`
  crossings, and the ratio summed over all test reticles.

`score` below does both. To see that it works, we score the crudest possible rule first: "a hit
map with any hit holds one crossing".

In [ ]:
def plane_sums(D, mask, est):
    # per sensor plane (shower, module, layer) of the selected reticles: estimate sum, truth sum, plane key
    key = (D['super_evt'][mask].astype(np.int64) * 64 + D['module'][mask]) * 256 + D['layer'][mask]
    uk, inv = np.unique(key, return_inverse=True)
    inv = inv.ravel()
    return np.bincount(inv, est, len(uk)), np.bincount(inv, D['n_frac'][mask], len(uk)), uk


def score(est, D=None, mask=None):
    # bias and rms per truth bin (mc.binned), and the ratio estimate / truth summed over sensor planes
    D = D1 if D is None else D
    mask = TEST if mask is None else mask
    truth = D['n_frac'][mask]
    assert len(est) == len(truth), "one estimate per selected reticle"
    se, st, _ = plane_sums(D, mask, est)
    ok = st >= PLANE_MIN
    ratio = se[ok] / st[ok]
    return {'bins': mc.binned(est, truth), 'total': est.sum() / truth.sum(),
            'plane_mean': ratio.mean(), 'plane_std': ratio.std(), 'n_planes': int(ok.sum())}


def show(scores, names, what=('rms', 'bias'), totals=True):
    # one line per estimator and quantity; '-' where a bin is empty; with totals, the summed ratios
    first = scores[names[0]]['bins']
    print(f"{'true count':27s}" + ''.join(f"{r['bin']:>8s}" for r in first))
    print(f"{'reticles':27s}" + ''.join(f"{r['N']:8,d}" for r in first))
    for w in what:
        for nm in names:
            vals = ''.join('       -' if r[w] is None else f'{r[w]:8.2f}' for r in scores[nm]['bins'])
            print(f'{w:4s} {nm:22s}{vals}')
    if totals:
        for nm in names:
            s = scores[nm]
            print(f"{nm:22s} summed over the reticles: estimate / truth = {s['total']:.3f}; per plane "
                  f"({s['n_planes']} planes with at least {PLANE_MIN} crossings): {s['plane_mean']:.3f} +- {s['plane_std']:.3f}")


P, SC = {}, {}                  # test estimates and their scores, per estimator
one = (VT['n_hits'] > 0).astype(float)
SC['one per map'] = score(one)
show(SC, ['one per map'])

tb = np.rint(TRUTH_T)
d = one[tb == 1] - TRUTH_T[tb == 1]
r1 = next(r for r in SC['one per map']['bins'] if r['bin'] == '1')
assert np.isclose(r1['rms'], np.sqrt(np.mean(d * d))) and np.isclose(r1['bias'], d.mean()), "score disagrees"
print(f"\ncheck: in the bin '1', score gives the rms and bias computed directly ({r1['rms']:.4f}, {r1['bias']:.4f})")
se, st, _ = plane_sums(D1, TEST, one)
assert np.isclose(st.sum(), TRUTH_T.sum()) and np.isclose(se.sum(), one.sum()), "plane sums lose reticles"
print(f"check: the {len(st):,} test planes hold all test crossings and all estimates")

"One per map" is unbiased only near one crossing; in every denser bin its bias is about minus the
number of extra crossings, and its rms grows with it. This is the scale against which the
estimators below should be read. Note also how few test reticles the dense bins hold: the rms
there rests on a few hundred reticles from 40 showers.

## 3. Estimator 1: count the hit pixels

The simplest rule that uses more than "any hit": for each number of hit pixels `h`, the mean true
count of the training reticles with `h` hit pixels. The estimate of a test reticle is the entry
of its `h`. We build the table with numpy and draw it with the spread of the truth at each `h`
(the band holds the central 68% of the training reticles). The right panel shows how many training
reticles each entry rests on.

In [ ]:
def table_by_value(x, y):
    # per integer value v of x: count, mean of y, and the 16% and 84% quantiles of y (NaN where empty)
    vmax = int(x.max())
    cnt = np.bincount(x, minlength=vmax + 1)
    s = np.bincount(x, y, vmax + 1)
    mean = np.full(vmax + 1, np.nan)
    mean[cnt > 0] = s[cnt > 0] / cnt[cnt > 0]
    ys = y[np.lexsort((y, x))]
    start = np.r_[0, np.cumsum(cnt)[:-1]]
    q16, q84 = np.full(vmax + 1, np.nan), np.full(vmax + 1, np.nan)
    for v in np.flatnonzero(cnt):
        q16[v], q84[v] = np.percentile(ys[start[v]:start[v] + cnt[v]], [16, 84])
    return cnt, mean, q16, q84


def draw_table(axs, cnt, mean, q16, q84, xlabel, diagonal=False):
    v = np.flatnonzero(cnt)
    full = cnt[v] >= M
    ax = axs[0]
    ax.fill_between(v, q16[v], q84[v], color='0.8', lw=0, label='central 68% of the training reticles')
    ax.plot(v[full], mean[v][full], 'o', ms=3, color='k', label=f'mean (entries with at least {M} reticles)')
    ax.plot(v[~full], mean[v][~full], 'o', ms=4, mfc='none', color='C3', label=f'mean (fewer than {M})')
    if diagonal:
        ax.plot([0, v.max()], [0, v.max()], '-', color='C0', lw=1, label='one crossing per cluster')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('true count n_frac')
    ax.legend(fontsize=9, loc='upper left')
    axs[1].bar(v, cnt[v], width=1.0, color='0.5')
    axs[1].axhline(M, color='C3', lw=1)
    axs[1].set_yscale('log')
    axs[1].set_xlabel(xlabel)
    axs[1].set_ylabel('training reticles')


fit = training(TRAIN)
V_TRAIN = mc.view([D1], [fit])
cnt_h, mean_h, q16_h, q84_h = table_by_value(D1['n_hits'][fit], D1['n_frac'][fit])
H_MAX = len(cnt_h) - 1
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={'width_ratios': [3, 2]}, constrained_layout=True)
draw_table(axs, cnt_h, mean_h, q16_h, q84_h, 'hit pixels h')
axs[0].set_title('the hit-count table, training showers')
plt.show()
for h in (1, 3, 5, 10, 20, 40, 80):
    print(f"h = {h:3d}: mean true count {mean_h[h]:6.2f}, central 68% {q16_h[h]:6.2f} to {q84_h[h]:6.2f}, "
          f"{cnt_h[h]:7,d} training reticles")
print(f"largest h in training: {H_MAX}; h = 0 (charge but no hit pixel): mean true count {mean_h[0]:.3f} "
      f"in {cnt_h[0]:,} reticles")

Up to ten hit pixels the table stays below two crossings, while the band reaches from a small
share of a crossing to a few crossings: one crossing alone can light one pixel or many (notebook
10, section 4). Above that the table rises steeply, because in dense reticles each crossing
lights fewer pixels (notebook 10, section 7). The band is wide everywhere, and the entries at the
far right rest on a handful of reticles.

**The study code's version.** `mc.Estimators(['hits'])` builds the same table with `mc.Lookup1D`
and differs in three places:

- an entry with fewer than `M` training reticles takes its neighbours on both sides, one step at a
  time, until the window holds `M` reticles; its value is the window's mean count times `h` divided
  by the window's mean `h`, so a window that can grow only to one side keeps the slope;
- above the largest `h` seen in training, the last entry is scaled in proportion to `h`;
- an empty hit map gives 0. A reticle with charge but no hit pixel looks exactly like the empty
  reticles that are not stored at all, so no rule can give it anything else; its crossings are
  lost to every binary estimator.

The check: our table equals mip_counting's wherever an entry stands alone.

In [ ]:
EST = {}                                                     # fitted estimators, by name
EST['hits'] = mc.Estimators(['hits']).fit(V_TRAIN, 'n_frac')
tab = EST['hits'].hits.table
full = cnt_h >= M
dmax = np.abs(tab[full] - mean_h[full]).max()
assert dmax < 1e-9, f"our table differs from mc.Lookup1D by {dmax}"
print(f"check: on the {int(full.sum())} values of h with at least {M} training reticles, our table equals "
      f"mc.Lookup1D (largest difference {dmax:.1e})")
sparse = np.flatnonzero(~full)
print(f"pooled entries: {len(sparse)} values of h, from {sparse.min()} up; for example")
for h in sparse[:4]:
    print(f"  h = {h}: {cnt_h[h]} training reticles, own mean {mean_h[h]:.2f}, mip_counting {tab[h]:.2f}")

P['hits'] = EST['hits'].predict(VT)['hits']
h_t = VT['n_hits']
alone = (h_t > 0) & (h_t <= H_MAX) & full[np.minimum(h_t, H_MAX)]
assert np.allclose(P['hits'][alone], mean_h[h_t[alone]]) and np.all(P['hits'][h_t == 0] == 0), \
    "mip_counting's estimate is not our table"
print(f"check: on the {alone.mean():.2%} of the test reticles whose h has a full entry, the estimate is our "
      f"table; empty maps ({np.mean(h_t == 0):.2%} of the test reticles, holding {TRUTH_T[h_t == 0].sum() / TRUTH_T.sum():.2%} "
      f"of the test crossings) get 0")
SC['hits'] = score(P['hits'])
print()
show(SC, ['hits'])

At one crossing the hit count is off by more than half a crossing (rms), because a single
crossing can light very different numbers of pixels.

The bias needs a word. The table gives, for each `h`, the mean truth of the reticles with that
`h`: it is unbiased in every bin of `h`. Binned by the **truth** instead, its estimates are pulled
toward the typical reticle with the same `h`: too high for the reticles of the bin "0" and too low
from two crossings up, because a reticle with more crossings than usual for its `h` lies above the
mean of the reticles that share its `h`, and gets that mean. Every estimator fitted as "mean truth
given what is seen" shows this pull, more strongly the wider its band. Summed over many reticles the pulls
cancel on average: the ratio over all test reticles is close to one, while single planes scatter
around it (the printed standard deviation).

## 4. Estimator 2: count the clusters

A **cluster** is a set of hit pixels that touch, at an edge or at a corner ("8-connected": each
pixel has 8 neighbours). In a sparse reticle a crossing usually makes one cluster, so the cluster
count should be a better count than the hit pixels.

First by hand on four test reticles, with `scipy.ndimage.label`: it gives every cluster a number,
and we colour each number. The title compares our count with `mc.clusters`.

In [ ]:
PALETTE = np.vstack([plt.get_cmap(n).colors for n in ('tab20', 'tab20b')])[:, :3]


def draw_labels(ax, lab, title):
    # a label image (0 = no hit), each cluster in its own colour
    rgb = np.ones(lab.shape + (3,))
    on = lab > 0
    rgb[on] = PALETTE[(lab[on] - 1) % len(PALETTE)]
    ax.imshow(rgb, origin='lower', interpolation='nearest', extent=EXT)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('v [um]', fontsize=9)
    ax.set_ylabel('u [um]', fontsize=9)
    ax.tick_params(labelsize=8)


EIGHT = np.ones((3, 3), int)          # the structure of 8-connectivity
rng = np.random.default_rng(SEED + 1)
ex4 = [pick_near(TEST & on_axis, D1['n_frac'], t, rng) for t in (2, 5, 12, 30)]
fig, axs = plt.subplots(2, 2, figsize=(12, 5.2), constrained_layout=True)
for ax, i in zip(axs.flat, ex4):
    lab, n = ndimage.label(HITS1[i], structure=EIGHT)
    draw_labels(ax, lab, f"n_frac = {D1['n_frac'][i]:.1f}: {n} clusters by hand, {D1['n_clusters'][i]} in mc.clusters")
plt.show()

In the sparse maps each colour is about one crossing; in the dense ones a single colour covers
several. Now the same count for all K1 reticles at once. Labelling them one by one would be slow,
so we stack the hit maps into one 3-D array and label it with a structure that connects pixels
within a hit map and never across two. Then we check that we find the same hit pixels, the same
clusters and the same count per reticle as `mc.clusters`.

In [ ]:
t0 = time.time()
st3 = np.zeros((3, 3, 3), int)
st3[1] = 1                                   # 8-connected within one hit map, never between two
lab, n_lab = ndimage.label(HITS1, structure=st3)
flat = lab.reshape(-1)
MY_IDX = np.flatnonzero(flat)                # flat index of every hit pixel
MY_LAB = flat[MY_IDX].astype(np.int64) - 1   # its cluster, 0 .. n_lab - 1
del lab, flat
S = NY * NX
pix_ret = MY_IDX // S
lo = np.full(n_lab, np.iinfo(np.int64).max)
hi = np.full(n_lab, -1)
np.minimum.at(lo, MY_LAB, pix_ret)
np.maximum.at(hi, MY_LAB, pix_ret)
assert np.array_equal(lo, hi), "a cluster spans two hit maps"
MY_RET = lo                                  # the reticle of each cluster
my_n_cl = np.bincount(MY_RET, minlength=D1['n'])
pix = D1['pix']
assert np.array_equal(MY_IDX, pix['idx']), "our hit pixels differ from mc.clusters"
pairs = np.unique(MY_LAB * (len(D1['cl']['size']) + 1) + pix['cl'])
assert len(pairs) == n_lab == len(D1['cl']['size']), "our clusters are not those of mc.clusters"
bad = my_n_cl != D1['n_clusters']
assert not bad.any(), f"our cluster count differs from mc.clusters in {int(bad.sum()):,} reticles"
print(f"labelled {n_lab:,} clusters in {len(MY_IDX):,} hit pixels in {time.time() - t0:.1f} s")
print(f"check: same hit pixels, same {n_lab:,} clusters (pixel by pixel), and the same count in all "
      f"{D1['n']:,} K1 reticles as mc.clusters")

The cluster-count table is built exactly like the hit-count table: the mean true count of the
training reticles with `c` clusters. The blue line is "one crossing per cluster". The printout
below the figure gives clusters per crossing in each bin of the true count, on the training
reticles. Look for where the table leaves the line.

In [ ]:
cnt_c, mean_c, q16_c, q84_c = table_by_value(D1['n_clusters'][fit], D1['n_frac'][fit])
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={'width_ratios': [3, 2]}, constrained_layout=True)
draw_table(axs, cnt_c, mean_c, q16_c, q84_c, 'clusters c', diagonal=True)
axs[0].set_title('the cluster-count table, training showers')
plt.show()

nf_tr, nc_tr = D1['n_frac'][fit], D1['n_clusters'][fit]
tb_tr = np.rint(nf_tr)
CL_PER_CROSSING = {}
print("true count            " + ''.join(f"{mc._bins_label(lo, hi):>8s}" for lo, hi in mc.TRUTH_BINS[1:]))
line = ''
for lo, hi in mc.TRUTH_BINS[1:]:
    s = (tb_tr >= lo) & ((tb_tr <= hi) if hi is not None else True)
    CL_PER_CROSSING[mc._bins_label(lo, hi)] = nc_tr[s].sum() / nf_tr[s].sum() if s.any() else np.nan
    line += '       -' if not s.any() else f'{nc_tr[s].sum() / nf_tr[s].sum():8.2f}'
print("clusters per crossing " + line)
print(f"the table at 1, 2 and 3 clusters: {mean_c[1]:.2f}, {mean_c[2]:.2f}, {mean_c[3]:.2f} crossings")
for lo, hi in ((6, 8), (9, 12), (13, 20)):
    s = (tb_tr >= lo) & (tb_tr <= hi)
    c_ = int(np.rint(nc_tr[s].mean()))
    print(f"true {lo}-{hi}: {nf_tr[s].mean():.1f} crossings, {nc_tr[s].mean():.1f} clusters; table at {c_}: {mean_c[c_]:.1f}")

EST['clusters'] = mc.Estimators(['clusters']).fit(V_TRAIN, 'n_frac')
full_c = cnt_c >= M
dmax = np.abs(EST['clusters'].clusters.table[full_c] - mean_c[full_c]).max()
assert dmax < 1e-9, f"our cluster table differs from mc.Lookup1D by {dmax}"
print(f"\ncheck: on the {int(full_c.sum())} values of c with at least {M} training reticles, our table equals "
      f"mc.Lookup1D (largest difference {dmax:.1e})")
P['clusters'] = EST['clusters'].predict(VT)['clusters']
SC['clusters'] = score(P['clusters'])
show(SC, ['hits', 'clusters'])
rms = {nm: [r['rms'] for r in SC[nm]['bins']] for nm in ('hits', 'clusters')}
labels = [r['bin'] for r in SC['hits']['bins']]
better = [b for b, a, c in zip(labels, rms['hits'], rms['clusters']) if a is not None and c < a]
print(f"\nthe cluster count has the smaller rms in the bins {', '.join(better)}")

In sparse reticles the cluster count is the best simple rule: its rms at one crossing is about
half that of the hit count, and it has the smaller rms of the two up to 6 to 8 crossings (the
printed list; at 13 to 20 both are poor). In the sparsest bin a crossing makes slightly more than one cluster on average,
and the table at one cluster is below one crossing: a crossing can break into separate pieces,
and a cluster can hold only the share of a crossing that has most of its charge in the
neighbouring reticle. Section 5 measures both.

The clusters per crossing fall steadily as the density rises, and in the densest bins most
crossings no longer have a cluster of their own. The table gives a reticle the typical truth of
the training reticles with its number of clusters, the pull of section 3. At one cluster that
typical reticle holds less than one crossing (broken crossings and edge pieces, above). At a
moderate number of clusters it holds more crossings than a reticle whose crossings are still
apart, because denser reticles whose clusters merged have that many clusters too (printed above).
So the table overestimates the moderately dense reticles (positive bias from 6 to 20 crossings)
and gives far too few in the densest. Notebook 10 (section 7) drew every crossing alone and found
that the fall of the clusters per crossing is mostly clusters touching, not crossings lighting
fewer pixels.

## 5. Estimator 3: cluster shapes

The cluster count throws away what a cluster looks like. A cluster of one or two pixels is
probably one crossing, or a piece of one; a large, filled blob probably holds several. The third
estimator gives every cluster a value from a **dictionary of shapes** and adds up the values of the
clusters of the reticle.

**The shape of a cluster** is three numbers: its **size** (hit pixels), and the **width** and
**height** of its bounding box (columns along v and rows along u). Several pixel patterns share one
shape: three pixels in a 2 x 2 box form an L in four orientations.

**The true content of a cluster.** To fill the dictionary we need, for every training cluster,
the number of crossings it holds. The file keeps one row for every crossing in every reticle it
touches: its charge there and the pixel where it left most of that charge in this reticle (its
*brightest pixel*). `mc.cluster_truth` (called by `mc.prepare`) defines:

- **strict content** (`t_n_frac`): the sum, over the rows whose brightest pixel is a hit pixel of
  the cluster, of the row's share of `n_frac` (its share of the crossing's charge);
- some rows have their brightest pixel off: a crossing with only a small share in this reticle,
  below the threshold on every pixel, or one whose brightest pixel fell below the threshold after
  noise. They belong to no cluster;
- **assigned content** (`a_n_frac`): the strict content plus those rows, each given to the cluster
  with the nearest hit pixel in its reticle. In a reticle without any hit pixel they stay lost.

The dictionary is fitted on the assigned content, so that the clusters of a reticle add up to its
truth. The picture shows one test reticle: top its charge, with a white dot on the brightest pixel
of each crossing; bottom its clusters, numbered; the printout gives the strict and the assigned
content of each.

In [ ]:
T1, W1, _ = mc.truths(R1, THRESHOLD)
assert np.allclose(T1['n_frac'], D1['n_frac']), "truths and prepare disagree"
W_ROW = W1['n_frac']                                 # the n_frac weight of every particle row
rows = np.flatnonzero(W_ROW > 0)
rflat = R1['part_reticle'][rows] * S + R1['part_iu'][rows].astype(np.int64) * NX + R1['part_iv'][rows]
k = np.minimum(np.searchsorted(pix['idx'], rflat), len(pix['idx']) - 1)
row_hit = pix['idx'][k] == rflat                 # is the row's brightest pixel a hit pixel?
cl = D1['cl']
strict_mine = np.bincount(pix['cl'][k[row_hit]], W_ROW[rows][row_hit], len(cl['size']))
dmax = np.abs(strict_mine - cl['t_n_frac']).max()
assert dmax < 1e-9, f"our strict content differs from mc.cluster_truth by {dmax}"
print(f"check: the strict content rebuilt from the brightest pixels equals mc.cluster_truth for all "
      f"{len(cl['size']):,} clusters (largest difference {dmax:.1e})")


def cluster_label_image(D, i):
    # the clusters of reticle i as a label image (1, 2, ... in the order of D['cl'])
    p = D['pix']
    a, b = np.searchsorted(p['ret'], [i, i + 1])
    first = np.searchsorted(D['cl']['reticle'], i)
    lab = np.zeros((NY, NX), int)
    lab[p['iy'][a:b], p['ix'][a:b]] = p['cl'][a:b] - first + 1
    return lab, first


# a test reticle with a few crossings and at least one row whose brightest pixel is off
off_ret = np.unique(R1['part_reticle'][rows][~row_hit])
cand = np.intersect1d(off_ret, np.flatnonzero(TEST & on_axis & (D1['n_frac'] > 3) & (D1['n_frac'] < 7)))
i = int(np.random.default_rng(SEED + 2).choice(cand))
lab, first = cluster_label_image(D1, i)
mine = rows[R1['part_reticle'][rows] == i]
fig, axs = plt.subplots(2, 1, figsize=(10, 6.6), constrained_layout=True)
q = img[i]
im = axs[0].imshow(np.where(q >= NOISE, q, np.nan), origin='lower', interpolation='nearest', extent=EXT,
                   norm=LogNorm(vmin=NOISE, vmax=1e5), cmap='viridis')
axs[0].plot((R1['part_iv'][mine] + 0.5) * PX, (R1['part_iu'][mine] + 0.5) * PY, 'o', ms=6, mfc='white', mec='k', mew=0.6)
axs[0].set_title(f"charge: n_frac = {D1['n_frac'][i]:.2f}, {len(mine)} counted rows (white dots: brightest pixels)",
                 fontsize=10)
fig.colorbar(im, ax=axs, label='charge of the top panel [e]', fraction=0.03, shrink=0.5)
draw_labels(axs[1], lab, f"{lab.max()} clusters, numbered as in the printout below "
                         f"(cross: a brightest pixel that is not hit)")
axs[0].set_xlabel(axs[1].get_xlabel())
axs[0].set_ylabel(axs[1].get_ylabel())
in_cl = np.flatnonzero(cl['reticle'] == i)           # in the order of the label image
for k_, c in enumerate(in_cl, 1):
    axs[1].text(cl['mean_x_um'][c], cl['mean_y_um'][c], str(k_), fontsize=8, ha='center', va='center',
                bbox=dict(fc='white', ec='0.3', lw=0.5, alpha=0.9, pad=1.2))
hit_mine = HITS1[i][R1['part_iu'][mine], R1['part_iv'][mine]]
axs[1].plot((R1['part_iv'][mine][~hit_mine] + 0.5) * PX, (R1['part_iu'][mine][~hit_mine] + 0.5) * PY, 'x', ms=9,
            color='k', mew=2)
plt.show()
print("cluster   size   box (v x u)   strict   assigned")
for k_, c in enumerate(in_cl, 1):
    print(f"{k_:7d}   {cl['size'][c]:4d}   {cl['width'][c]:5d} x {cl['height'][c]:<5d}  {cl['t_n_frac'][c]:6.2f}   {cl['a_n_frac'][c]:8.2f}")
print(f"rows of this reticle: {len(mine)}, of which {int((~hit_mine).sum())} with the brightest pixel not hit "
      f"(n_frac weight {W_ROW[mine][~hit_mine].sum():.2f}); clusters: strict sum {cl['t_n_frac'][in_cl].sum():.2f}, "
      f"assigned sum {cl['a_n_frac'][in_cl].sum():.2f}, truth {D1['n_frac'][i]:.2f}")

The crosses mark rows whose brightest pixel is off; the assigned content moves their share to a
nearby cluster. Clusters without any brightest pixel have a strict content of zero: they are
pieces of a crossing whose brightest pixel lies in another cluster.

Two checks over all K1 reticles: the strict contents of a reticle's clusters plus the weight of
its rows whose brightest pixel is off give its truth; the assigned contents plus the lost weight
give its truth. We rebuild the assigned contents and the lost weight from the rows first and
compare them with `mc.cluster_truth`.

In [ ]:
strict_sum = np.bincount(cl['reticle'], cl['t_n_frac'], D1['n'])
off_w = np.bincount(R1['part_reticle'][rows][~row_hit], W_ROW[rows][~row_hit], D1['n'])
lost = D1['lost']['n_frac']

# the assigned content rebuilt: a row whose brightest pixel is off goes to the cluster of the nearest hit
# pixel of its reticle (on a tie the first in row order, as in mc.cluster_truth); without a hit pixel it is lost
assigned_mine = np.bincount(pix['cl'][k[row_hit]], W_ROW[rows][row_hit], len(cl['size']))
un = rows[~row_hit]
has = D1['n_hits'][R1['part_reticle'][un]] > 0
lost_mine = np.bincount(R1['part_reticle'][un[~has]], W_ROW[un[~has]], D1['n'])
yy, xx = np.divmod(np.arange(S), NX)
u_ = un[has]
for a in range(0, len(u_), 20_000):
    uu = u_[a:a + 20_000]
    ret_ = R1['part_reticle'][uu]
    d2_ = (((xx - R1['part_iv'][uu][:, None].astype(np.int64)) * PX) ** 2
           + ((yy - R1['part_iu'][uu][:, None].astype(np.int64)) * PY) ** 2)
    d2_ = np.where(HITS1[ret_].reshape(len(uu), S), d2_, np.inf)
    np.add.at(assigned_mine, pix['cl'][np.searchsorted(pix['idx'], ret_ * S + d2_.argmin(1))], W_ROW[uu])
da = np.abs(assigned_mine - cl['a_n_frac']).max()
dl = np.abs(lost_mine - lost).max()
assert da < 1e-9 and dl < 1e-9, f"the rebuilt assigned contents or lost weight differ from mc.cluster_truth ({da}, {dl})"

d1 = np.abs(strict_sum + off_w - D1['n_frac']).max()
d2 = np.abs(np.bincount(cl['reticle'], assigned_mine, D1['n']) + lost_mine - D1['n_frac']).max()
assert d1 < 1e-6 and d2 < 1e-6, f"cluster contents do not add up ({d1}, {d2})"
tot = D1['n_frac'].sum()
print(f"check: per reticle, strict contents + weight of rows not hit = n_frac, and assigned contents + lost = "
      f"n_frac (largest differences {d1:.1e}, {d2:.1e}); the assigned contents and the lost weight, rebuilt "
      f"from the rows, equal mc.cluster_truth's (largest differences {da:.1e}, {dl:.1e})")
print(f"of the {tot:,.0f} crossings: {strict_sum.sum() / tot:.2%} in a cluster by their brightest pixel, "
      f"{off_w.sum() / tot:.2%} with the brightest pixel off, of which {lost.sum() / tot:.2%} in reticles without "
      f"a hit pixel")

**The dictionary.** For every shape seen in the training clusters: how many clusters have it,
the mean of their assigned content, and its standard deviation. The figure draws the most common
shapes (top row) and, in the bottom row, the most common shapes whose mean content is at least 1.5
crossings. Each picture is the most frequent pixel pattern of that shape, drawn with the true
2:1 pixel shape and all at the same scale. In the bottom row that pattern is only a small part of
its shape's clusters. Above each: the shape (size, width x height), its share
of the training clusters (top) or its number of training clusters (bottom), and the mean content
with its standard deviation.

In [ ]:
cfit = fit[cl['reticle']]                         # the clusters of training reticles
key = (cl['size'].astype(np.int64) * 64 + cl['width']) * 64 + cl['height']
KEYS, kinv, KCNT = np.unique(key[cfit], return_inverse=True, return_counts=True)
kinv = kinv.ravel()
y_cl = cl['a_n_frac'][cfit]
KSUM = np.bincount(kinv, y_cl, len(KEYS))
KMEAN = KSUM / KCNT
KSD = np.sqrt(np.maximum(np.bincount(kinv, y_cl * y_cl, len(KEYS)) / KCNT - KMEAN ** 2, 0))
K_SIZE, K_W, K_H = KEYS // 4096, KEYS // 64 % 64, KEYS % 64
n_cl_train = int(cfit.sum())
order = np.argsort(-KCNT, kind='stable')
TOP = order[:10]
multi = order[(KMEAN[order] >= 1.5) & (KCNT[order] >= 100)][:8]
print(f"{n_cl_train:,} training clusters in {len(KEYS):,} shapes; the 10 most common hold "
      f"{KCNT[TOP].sum() / n_cl_train:.1%} of them, the shapes with at least {M} clusters "
      f"{KCNT[KCNT >= M].sum() / n_cl_train:.1%}")

# the most frequent pixel pattern of each drawn shape: the pixels of a cluster as bits of its bounding box
want = np.zeros(len(KEYS), bool)
want[np.r_[TOP, multi]] = True
pc = pix['cl']
sel_cl = cfit & np.isin(key, KEYS[want])
sel_pix = sel_cl[pc]
dx = pix['ix'][sel_pix] - cl['x0'][pc[sel_pix]]
dy = pix['iy'][sel_pix] - cl['y0'][pc[sel_pix]]
bit = dy * cl['width'][pc[sel_pix]] + dx
assert bit.max() < 52, "a drawn shape is too large for an exact pattern code"
code_cl = np.bincount(pc[sel_pix], 2.0 ** bit, len(cl['size'])).astype(np.int64)
PATTERN, PATTERN_SHARE = {}, {}
for j in np.flatnonzero(want):
    codes = code_cl[cfit & (key == KEYS[j])]
    u, c = np.unique(codes, return_counts=True)
    PATTERN[j] = int(u[np.argmax(c)])
    PATTERN_SHARE[j] = c.max() / c.sum()

WIN_W = max(int(K_W[want].max()), 5) + 1           # the common window of the pictures, in pixels
WIN_H = max(int(K_H[want].max()), 6) + 2


def draw_shape(ax, j, top):
    w_, h_ = int(K_W[j]), int(K_H[j])
    bits = (PATTERN[j] >> np.arange(w_ * h_)) & 1
    pat = bits.reshape(h_, w_)
    x0, y0 = (WIN_W - w_) / 2 * PX, (WIN_H - h_) / 2 * PY
    ax.imshow(pat, origin='lower', interpolation='nearest', extent=(x0, x0 + w_ * PX, y0, y0 + h_ * PY),
              cmap=ListedColormap(['white', '0.15']), vmin=0, vmax=1)
    for xx in np.arange(w_ + 1):
        ax.plot([x0 + xx * PX] * 2, [y0, y0 + h_ * PY], color='0.75', lw=0.5)
    for yy in np.arange(h_ + 1):
        ax.plot([x0, x0 + w_ * PX], [y0 + yy * PY] * 2, color='0.75', lw=0.5)
    ax.set_xlim(0, WIN_W * PX)
    ax.set_ylim(0, WIN_H * PY)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    head = f"{KCNT[j] / n_cl_train:.1%} of clusters" if top else f"{KCNT[j]:,} clusters"
    ax.set_title(f"size {K_SIZE[j]}, {w_} x {h_}\n{head}\n{KMEAN[j]:.2f} +- {KSD[j]:.2f}", fontsize=9)


ncol = max(len(TOP), len(multi))
fig, axs = plt.subplots(2, ncol, figsize=(12, 4.1), constrained_layout=True)
for c_ in range(ncol):
    for r_, group in enumerate((TOP, multi)):
        if c_ < len(group):
            draw_shape(axs[r_, c_], group[c_], r_ == 0)
        else:
            axs[r_, c_].axis('off')
axs[0, 0].set_ylabel('most common', fontsize=10)
axs[1, 0].set_ylabel('mean >= 1.5', fontsize=10)
fig.suptitle(f'the shape dictionary: mean crossings per cluster +- standard deviation (pixels {PX:g} x {PY:g} um)',
             fontsize=11)
plt.show()
for j in np.r_[TOP, multi]:
    print(f"size {K_SIZE[j]:2d}, {K_W[j]} x {K_H[j]}: {KCNT[j]:7,d} clusters, mean {KMEAN[j]:.3f}, sd {KSD[j]:.3f}; "
          f"the drawn pattern is {PATTERN_SHARE[j]:.0%} of this shape")
one_px = cfit & (cl['size'] == 1)
a1 = cl['a_n_frac'][one_px]
print(f"\nsingle-pixel training clusters: {np.mean(a1 == 0):.0%} hold no crossing's brightest pixel, "
      f"{np.mean((a1 > 0) & (a1 < 0.99)):.0%} hold a share below one, {np.mean(a1 >= 0.99):.0%} one or more")

The most common shapes are small, and most of them hold less than one crossing on average. The
last printed line takes the single pixels apart: about half hold a whole crossing or more, nearly
as many hold only a share of a crossing that has the rest of its charge in another reticle, and a
few hold no brightest pixel at all (a piece of a crossing whose brightest pixel lies in another
cluster). The most common shapes that hold 1.5 crossings or more on average (bottom row) have
boxes of at least three by three pixels, and their spread is nearly as large as their mean: one
such shape covers clusters of a
single crossing and clusters of several. The dictionary cannot tell those apart; it gives their
average.

**The study code's version.** `mc.Estimators(['cluster_shape'])` treats rare shapes with care:

- every shape's mean is pulled toward the mean of all clusters of the same size (a size table
  built like the hit-count table), as if the shape had `M` extra clusters at the size's mean:
  value = (sum + `M` x size mean) / (count + `M`);
- then the values of all shapes of one size are scaled by one factor, so that the training clusters
  of that size keep their summed content;
- a shape never seen in training gets the size table's value.

For a shape with ten thousand or more training clusters both steps change almost nothing; with
1,000 to 9,999 they move it by up to 0.05 crossings, and rare shapes by far more (printed below).
The check below rebuilds the two steps with numpy and compares with mip_counting for every shape,
then shows by how much mip_counting departs from the plain means.

In [ ]:
EST['cluster_shape'] = mc.Estimators(['cluster_shape']).fit(V_TRAIN, 'n_frac')
e_cs = EST['cluster_shape']
uk_m, val_m = e_cs.shape[False]
assert np.array_equal(uk_m, KEYS), "mip_counting's dictionary has other shapes"
prior = e_cs.size.predict(K_SIZE)                          # the size table at each shape's size
shrunk = (KSUM + M * prior) / (KCNT + M)
g_cnt, g_sum = np.bincount(K_SIZE, KCNT), np.bincount(K_SIZE, KSUM)
g_den = np.bincount(K_SIZE, KCNT * shrunk)
scale = np.ones(len(g_cnt))
ok = (g_cnt >= M) & (g_den > 0)
scale[ok] = g_sum[ok] / g_den[ok]
rebuilt = shrunk * scale[K_SIZE]
dmax = np.abs(rebuilt - val_m).max()
assert dmax < 1e-9, f"the rebuilt dictionary differs from mip_counting by {dmax}"
print(f"check: pulling toward the size table and scaling per size, rebuilt with numpy, gives mip_counting's value "
      f"for all {len(KEYS):,} shapes (largest difference {dmax:.1e})")
print("\ntraining clusters per shape   shapes   share of clusters   largest |mip_counting - plain mean|")
for lo_, hi_ in ((1, M), (M, 100), (100, 1000), (1000, 10000), (10000, None)):
    s = (KCNT >= lo_) & ((KCNT < hi_) if hi_ else True)
    print(f"{lo_:>8,d} to {('' if hi_ is None else f'{hi_ - 1:,d}'):>8s}        {int(s.sum()):6,d}   "
          f"{KCNT[s].sum() / n_cl_train:17.2%}   {np.abs(val_m[s] - KMEAN[s]).max():20.4f}")
common = KCNT >= 10000
dcommon = np.abs(val_m[common] - KMEAN[common]).max()
assert dcommon < 0.005, f"mip_counting departs from the plain mean by {dcommon} on a common shape"
print(f"check: on the {int(common.sum())} shapes with at least 10,000 training clusters mip_counting and the plain "
      f"mean agree within {dcommon:.4f}")

For the common shapes mip_counting's dictionary is the plain mean. The pull matters only for rare
shapes, where the plain mean of a handful of clusters would be noise.

**Applied to the test showers.** The estimate of a reticle is the sum of the dictionary values of
its clusters. We do this with our plain dictionary (unseen shapes from mip_counting's size table)
and check that mip_counting's estimate is the same sum over its own values.

In [ ]:
tcl = VT['cl']
tkey = (tcl['size'].astype(np.int64) * 64 + tcl['width']) * 64 + tcl['height']
pos = np.minimum(np.searchsorted(KEYS, tkey), len(KEYS) - 1)
known = KEYS[pos] == tkey
size_val = e_cs.size.predict(tcl['size'])
plain_val = np.where(known, KMEAN[pos], size_val)
module_val = np.where(known, val_m[pos], size_val)
P['cluster_shape'] = e_cs.predict(VT)['cluster_shape']
module_sum = np.bincount(tcl['reticle'], module_val, VT['n'])
dmax = np.abs(module_sum - P['cluster_shape']).max()
assert dmax < 1e-9, f"mip_counting's estimate is not the sum of its dictionary values ({dmax})"
print(f"check: mip_counting's estimate of every test reticle is the sum of its clusters' dictionary values "
      f"(largest difference {dmax:.1e})")
print(f"test clusters with a shape never seen in training: {np.mean(~known):.3%}")
SC['cluster_shape'] = score(P['cluster_shape'])
SC['plain shape dictionary'] = score(np.bincount(tcl['reticle'], plain_val, VT['n']))
show(SC, ['hits', 'clusters', 'plain shape dictionary', 'cluster_shape'])
ahead = [r['bin'] for r, c in zip(SC['cluster_shape']['bins'], SC['clusters']['bins']) if r['N'] and r['rms'] < c['rms']]
print(f"\nthe shape dictionary has a smaller rms than the cluster count in the bins {', '.join(ahead)}")
rare = ~known | (KCNT[pos] < 100)                 # shapes with fewer than 100 training clusters
tdense = np.rint(TRUTH_T)[tcl['reticle']]
content = tcl['a_n_frac']
groups = {'1 to 5': (1, 5), '6 to 20': (6, 20), '21 or more': (21, 10 ** 6)}
sel = {g: (tdense >= lo_) & (tdense <= hi_) for g, (lo_, hi_) in groups.items()}
print(f"\ntest clusters, by the true count of their reticle {''.join(f'{g:>12s}' for g in groups)}")
print("shapes with fewer than 100 training clusters:")
print(f"  share of the clusters                           "
      + ''.join(f"{np.mean(rare[s]):12.1%}" for s in sel.values()))
print(f"  share of the content (crossings)                "
      + ''.join(f"{content[s & rare].sum() / content[s].sum():12.1%}" for s in sel.values()))
print(f"  dictionary value / content                      "
      + ''.join(f"{module_val[s & rare].sum() / content[s & rare].sum():12.2f}" for s in sel.values()))
print("shapes with 100 training clusters or more:")
print(f"  dictionary value / content                      "
      + ''.join(f"{module_val[s & ~rare].sum() / content[s & ~rare].sum():12.2f}" for s in sel.values()))

The shape dictionary is behind the cluster count up to three crossings and ahead of it in the
moderately dense bins (the printed list): there a large cluster counts for several crossings
instead of one. A cluster's value is an average over all training clusters of its shape, in
sparse and in dense reticles alike, and it does not know which kind of reticle it sits in. The
last printed table measures what that does, comparing each cluster's dictionary value with its
assigned content:

- in reticles of up to five crossings the common shapes get about what they hold, and the rare
  shapes (fewer than 100 training clusters) far more;
- from six crossings up, clusters of every kind get less than they hold, and the more so the
  denser the reticle: the same shape holds more crossings in a dense reticle than on average;
- in reticles of 21 crossings or more the rare shapes are a minority of the clusters but hold most
  of the crossings.

So the dictionary falls far too low in the densest bins, through common and rare shapes alike. The
pull toward the size table changes the rms by up to about a tenth: it lowers it from one to twelve
crossings and raises it in the bin 0 and from 13 crossings up.

### 5b. Would a library of exact shapes do better?

The dictionary groups the clusters by size, width and height, so several pixel patterns share one
entry. A stored library could keep every pattern instead. The **exact shape** of a cluster is the
set of its pixels relative to its bounding box: two clusters share an exact shape only if they are
the same pattern. The library holds the exact shapes of the training clusters. Its value for a
shape is the mean assigned content of its training clusters, pulled toward the size table as in
section 5 but with `M_EXACT` extra clusters instead of `M`, so that a shape seen a few times
already counts. A shape absent from the library gets the size table's value.

First, how many exact shapes there are: over the clusters of all 200 showers, and in the library.

In [ ]:
# the exact shape of every K1 cluster: the offsets of its pixels in its bounding box, as one key
e_ord = np.argsort(pix['cl'], kind='stable')      # pixels by cluster, within a cluster in the order of pix['idx']
e_cl = pix['cl'][e_ord]
e_rel = ((pix['iy'][e_ord] - cl['y0'][e_cl]) * NX + pix['ix'][e_ord] - cl['x0'][e_cl]).astype(np.int16)
assert np.all(np.diff(e_rel)[np.diff(e_cl) == 0] > 0), "the offsets within a cluster are not sorted"
e_start = np.r_[0, np.cumsum(cl['size'])[:-1]]
e_key = np.array([e_rel[a:a + s].tobytes() for a, s in zip(e_start, cl['size'])], dtype=object)
E_KEYS, e_inv, e_cnt = np.unique(e_key, return_inverse=True, return_counts=True)
e_inv = e_inv.ravel()
del e_ord, e_cl, e_rel, e_key
N_CL_ALL = len(cl['size'])
N90 = int(np.searchsorted(np.cumsum(np.sort(e_cnt)[::-1]), 0.9 * N_CL_ALL)) + 1
lib_n = np.bincount(e_inv[cfit], minlength=len(E_KEYS))          # training clusters per exact shape
print(f"{N_CL_ALL:,} clusters of all 200 showers in {len(E_KEYS):,} exact shapes; {N90:,} shapes cover 90% of the "
      f"clusters; the library: {int(np.count_nonzero(lib_n)):,} exact shapes in {n_cl_train:,} training clusters")

SIZE_CLASSES = ((1, 2), (3, 4), (5, 8), (9, 16), (17, 32), (33, 64), (65, None))
print("\n                       all 200 showers                        library (training clusters)")
print("cluster size     clusters   exact shapes   seen once      clusters   exact shapes   seen once")
TABLE_ALL = []
for lo_, hi_ in SIZE_CLASSES:
    s_all = (cl['size'] >= lo_) & ((cl['size'] <= hi_) if hi_ else True)
    s_lib = s_all & cfit
    once_all, once_lib = np.mean(e_cnt[e_inv[s_all]] == 1), np.mean(lib_n[e_inv[s_lib]] == 1)
    TABLE_ALL.append((len(np.unique(e_inv[s_all])), round(100 * once_all, 1)))
    print(f"{mc._bins_label(lo_, hi_):>12s}   {int(s_all.sum()):10,d}   {TABLE_ALL[-1][0]:12,d}   {once_all:9.1%}    "
          f"{int(s_lib.sum()):10,d}   {len(np.unique(e_inv[s_lib])):12,d}   {once_lib:9.1%}")
large = cl['size'] >= 17
print(f"all 200 showers: {len(np.unique(e_inv[cl['size'] <= 4])):,} exact shapes of up to 4 pixels; "
      f"{np.mean(e_cnt[e_inv[large]] == 1):.1%} of the clusters of 17 pixels or more have an exact shape seen once")

# the whole hit map as a key
has_hit = D1['n_hits'] > 0
packed = np.packbits(HITS1.reshape(D1['n'], -1), axis=1)[has_hit]
_, m_inv, m_cnt = np.unique(packed, axis=0, return_inverse=True, return_counts=True)
m_once = m_cnt[m_inv.ravel()] == 1
m_big = D1['n_hits'][has_hit] >= 10
print(f"\nwhole hit maps: {int(has_hit.sum()):,} K1 reticles with a hit pixel, {len(m_cnt):,} distinct maps; "
      f"{m_once.mean():.1%} of these reticles have a map seen nowhere else, {m_once[m_big].mean():.1%} of those with "
      f"at least 10 hit pixels")
del packed, m_inv

assert (N_CL_ALL, len(E_KEYS), N90) == (880356, 54751, 1088) and TABLE_ALL == \
       [(5, 0.0), (129, 0.0), (10886, 3.7), (36668, 48.6), (6378, 99.7), (652, 100.0), (33, 100.0)] and \
       (int(has_hit.sum()), len(m_cnt), round(100 * m_once.mean(), 1), round(100 * m_once[m_big].mean(), 1)) == \
       (571810, 242495, 37.6, 93.3), "the counts over all showers differ from those quoted in the text below"
print("check: the counts over all 200 showers are those quoted in the text below")

In [ ]:
M_EXACT = 5                                       # the pull of the library: 5 extra clusters at the size table's value
tsel = TEST[cl['reticle']]                        # the test clusters, in the order of VT['cl']
assert np.array_equal(cl['size'][tsel], tcl['size']) and np.array_equal(cl['a_n_frac'][tsel], tcl['a_n_frac']), \
    "the test clusters are not those of VT"
lib_sum = np.bincount(e_inv[cfit], cl['a_n_frac'][cfit], len(E_KEYS))
k_t = e_inv[tsel]
lib_val = (lib_sum[k_t] + M_EXACT * size_val) / (lib_n[k_t] + M_EXACT)    # size_val: the size table (section 5)
absent = lib_n[k_t] == 0
print(f"library value = (sum + {M_EXACT} x size table) / (count + {M_EXACT}); test clusters whose exact shape is absent "
      f"from the library: {absent.mean():.1%}, holding {content[absent].sum() / content.sum():.1%} of the test content")
print(f"by the true count of the reticle             {''.join(f'{g:>12s}' for g in groups)}")
print(f"  absent: share of the clusters              " + ''.join(f"{np.mean(absent[s]):12.1%}" for s in sel.values()))
print(f"  absent: share of the content               "
      + ''.join(f"{content[s & absent].sum() / content[s].sum():12.1%}" for s in sel.values()))

P['size table'] = np.bincount(tcl['reticle'], size_val, VT['n'])
P['exact-shape library'] = np.bincount(tcl['reticle'], lib_val, VT['n'])
for nm in ('size table', 'exact-shape library'):
    SC[nm] = score(P[nm])
print()
show(SC, ['size table', 'exact-shape library', 'plain shape dictionary', 'cluster_shape'], totals=False)

# how much the content varies among training clusters of one exact shape (shapes with at least 50 of them)
y_lib, k_lib, s_lib = cl['a_n_frac'][cfit], e_inv[cfit], cl['size'][cfit]
resid = y_lib - (lib_sum / np.maximum(lib_n, 1))[k_lib]
common_e = lib_n[k_lib] >= 50
print("\ntraining clusters of exact shapes with at least 50 training clusters:")
print("cluster size   clusters   mean content   rms around the mean of their shape")
SPREAD = {}
for lo_, hi_ in SIZE_CLASSES[:4]:
    s = common_e & (s_lib >= lo_) & (s_lib <= hi_)
    SPREAD[mc._bins_label(lo_, hi_)] = np.sqrt(np.mean(resid[s] ** 2))
    print(f"{mc._bins_label(lo_, hi_):>12s}   {int(s.sum()):8,d}   {y_lib[s].mean():12.2f}   "
          f"{SPREAD[mc._bins_label(lo_, hi_)]:34.2f}")

The library is small where it adds little and cannot be filled where it would matter. Up to 4
pixels there are only 134 exact shapes; from 17 pixels up, 99.7% of the clusters have an exact
shape seen nowhere else in the 200 showers. In the test showers the clusters whose shape is absent
from the library hold 6.1% of the content of the reticles of up to five crossings, and 63.9% in
reticles of 21 crossings or more. A common small shape does not fix the count either: clusters of
one exact shape of 3 to 4 pixels spread by 0.46 crossings around their shape's mean.

So the library scores like the tables of section 5: from four crossings up its rms lies between
that of the size table and that of `cluster_shape`, below that it is about equal to
`cluster_shape`, and it falls as far short in the dense bins (the bias rows). The whole hit map is
no better a key: 93.3% of the reticles with at least 10 hit pixels have a map seen nowhere else.
The trees of section 6 look up no shape; they read numbers that every hit map has, seen or not.

## 6. Estimator 4: boosted trees on 49 numbers

The tables use one or a few numbers of the hit map at a time. The last estimator uses 49 at once
(`mc.reticle_features`, the column `feat` of `mc.prepare`):

1. the number of hit pixels;
2. the number of clusters;
3. to 10. the number of clusters of size 1, 2, 3, 4, 5 to 6, 7 to 10, 11 to 20, and 21 or more
   (`mc.SIZE_BINS`);
11. the hit pixels on the border of the reticle;
12. the clusters that touch the border;
13. the size of the largest cluster;
14. the number of rows with a hit, and 15. the number of columns with a hit;
16. to 28. the hit pixels in each of the 13 rows;
29. to 49. the hit pixels in each of the 21 columns.

The border numbers matter because a cluster cut by the border may be a share of a crossing that
lies mostly in the neighbouring reticle. First the check: we rebuild all 49 numbers from our own
clusters of section 4 and the hit maps.

In [ ]:
n1 = D1['n']
size_my = np.bincount(MY_LAB, minlength=n_lab)
iy_p, ix_p = MY_IDX % S // NX, MY_IDX % NX
edge_p = (ix_p == 0) | (ix_p == NX - 1) | (iy_p == 0) | (iy_p == NY - 1)
edge_lab = np.bincount(MY_LAB[edge_p], minlength=n_lab) > 0
sb = np.searchsorted(np.array(mc.SIZE_BINS), size_my, side='right') - 1
rows_hit, cols_hit = HITS1.sum(axis=2), HITS1.sum(axis=1)
max_size = np.zeros(n1, np.int64)
np.maximum.at(max_size, MY_RET, size_my)
F = np.column_stack([HITS1.sum(axis=(1, 2)), my_n_cl,
                     np.bincount(MY_RET * len(mc.SIZE_BINS) + sb, minlength=n1 * len(mc.SIZE_BINS)).reshape(n1, -1),
                     np.bincount(pix_ret[edge_p], minlength=n1), np.bincount(MY_RET[edge_lab], minlength=n1),
                     max_size, (rows_hit > 0).sum(1), (cols_hit > 0).sum(1), rows_hit, cols_hit])
assert F.shape == D1['feat'].shape == (n1, 49), f"{F.shape} numbers, not 49 per reticle"
assert np.array_equal(F, D1['feat']), "our 49 numbers differ from mc.reticle_features"
print(f"check: our {F.shape[1]} numbers equal mc.reticle_features for all {n1:,} K1 reticles "
      f"(cluster size classes start at {', '.join(str(x) for x in mc.SIZE_BINS)} pixels)")
del F, rows_hit, cols_hit
j = ex4[1]
SB = list(mc.SIZE_BINS) + [None]
size_names = [f"size {mc._bins_label(lo_, None if hi_ is None else hi_ - 1)}" for lo_, hi_ in zip(SB[:-1], SB[1:])]
names_ = [size_names[c_ - 2] if nm.startswith('size_ge') else nm for c_, nm in enumerate(mc.FEATURES)]
print(f"\nthe first 15 numbers of the reticle with n_frac = {D1['n_frac'][j]:.1f} of section 4:")
print('  ' + ', '.join(f"{name}: {int(v)}" for name, v in zip(names_, D1['feat'][j])))

**Boosted decision trees.** A decision tree splits the training reticles by a sequence of yes/no
questions on these numbers ("more than 12 hit pixels?") into leaves, and predicts the mean count
of the leaf. Boosting adds trees one after another: each new tree is fitted to what the sum of the
previous trees still gets wrong, and is added with a small weight (0.1 here). The study code uses
scikit-learn's `HistGradientBoostingRegressor` with `mc.GBT_ITER` trees of at most 31 leaves, at
least `M` training reticles per leaf, and the squared error as the quantity to minimise. It
trains on the training reticles with at least one hit pixel (at most `mc.GBT_MAX_ROWS` of them,
a fixed random subset beyond that).

`gbt_layer` is the same model with the layer number as a 50th input. A sensor knows its layer; it
does not know where the shower axis is. Section 9 shows what the layer adds.

In [ ]:
t0 = time.time()
EST['gbt'] = mc.Estimators(['gbt', 'gbt_layer']).fit(V_TRAIN, 'n_frac')
n_rows = int(np.count_nonzero(V_TRAIN['n_hits'] > 0))
print(f"{mc.GBT_ITER} trees each for gbt and gbt_layer, trained in {time.time() - t0:.0f} s on "
      f"{min(n_rows, mc.GBT_MAX_ROWS):,} of the {n_rows:,} training reticles with a hit pixel")
gp = EST['gbt'].gbts['gbt'].get_params()
print(f"settings: {gp['max_iter']} trees, weight (learning rate) {gp['learning_rate']}, at most {gp['max_leaf_nodes']} "
      f"leaves, at least {gp['min_samples_leaf']} training reticles per leaf, loss {gp['loss']}; "
      f"{EST['gbt'].gbts['gbt'].n_features_in_} and {EST['gbt'].gbts['gbt_layer'].n_features_in_} inputs")
P.update(EST['gbt'].predict(VT))
for nm in ('gbt', 'gbt_layer'):
    SC[nm] = score(P[nm])
show(SC, ['hits', 'clusters', 'cluster_shape', 'gbt', 'gbt_layer'], what=('rms',))
show(SC, ['gbt', 'gbt_layer'], what=('bias',), totals=False)
FOUR = ('hits', 'clusters', 'cluster_shape', 'gbt')
GBT_BEST = [r['bin'] for b, r in enumerate(SC['gbt']['bins'])
            if r['N'] and min(FOUR, key=lambda nm: SC[nm]['bins'][b]['rms']) == 'gbt']
N_BINS_T = sum(1 for r in SC['gbt']['bins'] if r['N'])
print(f"\namong the four estimators, gbt has the smallest rms in {len(GBT_BEST)} of the {N_BINS_T} filled bins; "
      f"not in: {', '.join(r['bin'] for r in SC['gbt']['bins'] if r['N'] and r['bin'] not in GBT_BEST) or 'none'}")

The trees have the smallest rms of the four in nearly every bin (the printout names any
exception, and section 7b shows how much one block of 40 showers can move the rms). Their bias
stays within a fraction of a crossing up to 20 crossings; in the densest bins they are low. The
layer number helps a little in the dense bins. The tables use one or two numbers of the hit map
at a time; the trees use all 49 together, and no single table comes close to them over the whole
range.

## 7. All estimators side by side

The rms and the bias per bin of the true count, on the test showers. The second table sets the two
rows of section 5b, the size table alone and the library of exact shapes, beside the shape
dictionary and the trees.

In [ ]:
show(SC, list(NAMES))
best = []
for b, row in enumerate(SC['hits']['bins']):
    if row['N']:
        best.append((row['bin'], min(NAMES, key=lambda nm: SC[nm]['bins'][b]['rms'])))
print('\nsmallest rms: ' + ', '.join(f"{b}: {nm}" for b, nm in best))
print()
show(SC, ['size table', 'exact-shape library', 'cluster_shape', 'gbt'], what=('rms',), totals=False)

In the second table the trees have the smallest rms in every bin but 4-5, where the four rows are
close. Keeping every exact shape does not bring a table near the trees.

The same as a figure, relative to the true count: the rms divided by the mean truth of the bin
(left) and the bias divided by it (right). Look at where each line rises above the others.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
for nm in NAMES:
    rows_ = [r for r in SC[nm]['bins'] if r['N'] and r['truth'] >= 0.5]
    t_ = np.array([r['truth'] for r in rows_])
    axs[0].plot(t_, [r['rms'] / r['truth'] for r in rows_], STYLE[nm], marker='o', ms=3, color=COLOUR[nm], label=nm)
    axs[1].plot(t_, [r['bias'] / r['truth'] for r in rows_], STYLE[nm], marker='o', ms=3, color=COLOUR[nm], label=nm)
axs[1].axhline(0, color='0.5', lw=0.8)
for ax, yl in zip(axs, ('rms / mean truth', 'bias / mean truth')):
    ax.set_xscale('log')
    ax.set_xlabel('mean true count of the bin (n_frac)')
    ax.set_ylabel(yl)
    ax.grid(alpha=0.3)
axs[0].set_ylim(0, None)
axs[0].legend(fontsize=9)
fig.suptitle('test showers, per bin of the true count', fontsize=11)
plt.show()
rel_bins = [r['bin'] for r in SC['gbt']['bins'] if r['N'] and r['truth'] >= 0.5]
print("rms / mean truth    " + ''.join(f"{b:>7s}" for b in rel_bins))
for nm in NAMES:
    print(f"{nm:20s}" + ''.join(f"{r['rms'] / r['truth']:7.2f}" for r in SC[nm]['bins'] if r['N'] and r['truth'] >= 0.5))

Relative to the count, the trees are nearly flat up to about 20 crossings and better beyond. The
tables are not: the hit count is poor below ten crossings, the cluster count is good in sparse
and poor in dense reticles, and the shape dictionary is at its best in between.

**Estimate against truth**, reticle by reticle, as 2-D histograms on logarithmic axes (reticles
with a truth below 0.5 or an estimate below 0.5 fall outside). The grey line is estimate = truth.

In [ ]:
edges = np.geomspace(0.5, 150, 46)
fig, axs = plt.subplots(2, 3, figsize=(12, 7.6), constrained_layout=True)
for ax, nm in zip(axs.flat, NAMES):
    H, _, _ = np.histogram2d(TRUTH_T, P[nm], bins=[edges, edges])
    m_ = ax.pcolormesh(edges, edges, np.where(H > 0, H, np.nan).T, norm=LogNorm(vmin=1, vmax=1e4), cmap='viridis')
    ax.plot(edges, edges, color='0.5', lw=1)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(nm, fontsize=10)
    ax.set_xlabel('true count n_frac')
    ax.set_ylabel('estimate')
axs.flat[-1].axis('off')
fig.colorbar(m_, ax=axs.flat[-1], label='test reticles', location='left', fraction=0.15)
plt.show()
for nm in NAMES:
    out = np.mean((TRUTH_T >= 0.5) & (P[nm] < 0.5))
    print(f"{nm:14s}: {out:.2%} of the test reticles have a truth of at least 0.5 and an estimate below 0.5")

The hit-count table spreads widely at every truth. The cluster count sits on the line up to a few
crossings, then flattens: its estimates stop rising while the truth keeps rising. The shape dictionary and the trees follow the line
further; the trees have the narrowest band. The trees put a few per cent of the test reticles
below half a crossing although they hold at least half a crossing (printed: 3.05% for gbt, 3.19%
for gbt_layer, against 0.10 to 0.32% for the tables); those lie outside the plot.

### 7b. The write-up's table, reproduced

Section 6.1 of `docs/reticle_mip_counting.md` gives the rms per bin for single showers from five
blocks: each block of 40 showers in turn is estimated by models trained on the other four blocks,
so that every shower is estimated once by a model that never saw it. Our test showers are the
last of those blocks, with the same training showers and readout settings. Comparing one block of
40 showers with all 200 would need a tolerance that we cannot set in advance: the rms of a dense
bin is set by a few reticles with large errors.

So we repeat the write-up's procedure: four more fits, one for each other block. These fits use
test showers for training and are used only in the comparisons with the write-up (here and in
section 9); every other number of this notebook comes from the models trained on showers 0 to
159. `mc.check_unseen` checks every block.
The numbers of section 6.1 were read from the write-up when this notebook was built:

In [ ]:
# docs/reticle_mip_counting.md, sections 6.1 (rms per bin, 5 blocks) and 6.4 (layer groups)
WRITEUP = {'section_6_1': {'bins': ['1', '2', '3', '4-5', '6-8', '9-12', '13-20', '21-40', '41-80'],
                           'rms': {'hits': [0.57, 1.1, 1.81, 2.79, 4.18, 5.72, 7.43, 10.99, 13.01],
                                   'clusters': [0.32, 0.67, 1.0, 1.62, 3.44, 6.13, 7.11, 12.0, 34.87],
                                   'cluster_shape': [0.52, 0.84, 1.06, 1.36, 2.18, 3.76, 7.28, 15.48, 20.62],
                                   'gbt': [0.28, 0.59, 0.88, 1.2, 1.89, 2.98, 4.37, 6.52, 8.03]},
                           'reticles': [342325, 54697, 18765, 14931, 7273, 3099, 2592, 1881, 912]},
           'section_6_4': {'groups': [(0, 4), (5, 9), (10, 14), (15, 19), (20, 29)],
                           'ratio': {'hits': [0.72, 0.95, 1.0, 1.01, 1.12],
                                     'clusters': [0.72, 0.92, 1.03, 1.04, 1.01],
                                     'gbt': [0.89, 0.99, 1.01, 1.0, 1.01],
                                     'gbt_layer': [0.99, 1.0, 1.0, 1.0, 1.0]},
                           'share': [0.031, 0.214, 0.355, 0.279, 0.12],
                           'hits_per_crossing': [2.9, 3.5, 4.0, 4.4, 4.7]}}
W61 = WRITEUP['section_6_1']
NAMES_61 = ('hits', 'clusters', 'cluster_shape', 'gbt')
NAMES_5B = NAMES_61 + ('gbt_layer',)            # gbt_layer too: section 9 compares the layer groups
t0 = time.time()
PB = {nm: np.full(D1['n'], np.nan) for nm in NAMES_5B}
for f in range(N_BLOCKS):
    tr_f, ev_f = D1['fold'] != f, D1['fold'] == f
    mc.check_unseen(R1['n_events'], [D1], [tr_f], [D1], [ev_f], f'block {f}')
    if f == N_BLOCKS - 1:                       # the test block: the models of sections 3 to 6
        assert np.array_equal(tr_f, TRAIN), "the last block's training set is not ours"
        for nm in NAMES_5B:
            PB[nm][ev_f] = P[nm]
        continue
    pr = mc.Estimators(NAMES_5B).fit(mc.view([D1], [tr_f]), 'n_frac').predict(mc.view([D1], [ev_f]))
    for nm in NAMES_5B:
        PB[nm][ev_f] = pr[nm]
assert all(np.isfinite(PB[nm]).all() for nm in NAMES_5B), "a reticle without an estimate"
print(f"check: in each of the {N_BLOCKS} blocks no shower is both trained on and estimated; "
      f"four extra fits took {time.time() - t0:.0f} s")

ALL5 = {nm: {r['bin']: r for r in mc.binned(PB[nm], D1['n_frac'])} for nm in NAMES_61}
BLOCK = {nm: [{r['bin']: r for r in mc.binned(PB[nm][D1['fold'] == f], D1['n_frac'][D1['fold'] == f])}
              for f in range(N_BLOCKS)] for nm in NAMES_61}
n_all = [ALL5['hits'][b]['N'] for b in W61['bins']]
assert n_all == W61['reticles'], f"reticles per bin {n_all} differ from the write-up's {W61['reticles']}"
print(f"check: the reticles per bin over all 200 showers are those of the write-up ({', '.join(f'{x:,}' for x in n_all)})")

print("\nrms per bin    " + ''.join(f"{b:>8s}" for b in W61['bins']))
TOL = {}
for nm in NAMES_61:
    mine = np.array([ALL5[nm][b]['rms'] for b in W61['bins']])
    doc = np.array(W61['rms'][nm])
    blk = np.array([[BLOCK[nm][f][b]['rms'] for b in W61['bins']] for f in range(N_BLOCKS)])
    print(f"{nm}")
    print("  test block   " + ''.join(f"{x:8.2f}" for x in blk[-1]))
    print("  block min    " + ''.join(f"{x:8.2f}" for x in blk.min(0)))
    print("  block max    " + ''.join(f"{x:8.2f}" for x in blk.max(0)))
    print("  range / rms  " + ''.join(f"{x:8.2f}" for x in (blk.max(0) - blk.min(0)) / mine))
    print("  all 5 blocks " + ''.join(f"{x:8.2f}" for x in mine))
    print("  write-up     " + ''.join(f"{x:8.2f}" for x in doc))
    TOL[nm] = np.abs(mine - doc) / doc
for nm in ('hits', 'clusters', 'cluster_shape'):
    d_ = np.abs(np.array([ALL5[nm][b]['rms'] for b in W61['bins']]) - W61['rms'][nm]).max()
    assert d_ <= 0.006, f"{nm}: the 5-block rms differs from the write-up by up to {d_:.3f}"
    print(f"check: {nm}: the 5-block rms equals the write-up to its printed digits (largest difference {d_:.3f})")
b_g = W61['bins'][int(np.argmax(TOL['gbt']))]       # the bin of the largest difference
n_g = ALL5['gbt'][b_g]['N']
print(f"gbt: the 5-block rms differs from the write-up by up to {TOL['gbt'].max():.1%}, in the bin {b_g}; printed, "
      f"not asserted ({n_g:,} reticles in that bin over the five blocks; the statistical error of an rms from N "
      f"values is about 1/sqrt(2N) = {1 / np.sqrt(2 * n_g):.1%})")

The three tables are plain averages: refitted here they give the write-up's numbers to the
printed digits. The trees agree within a few per cent (printed) but not to the last digit, about
the size of the statistical error of the rms of that bin. We have not traced the difference, so this
comparison is printed and not asserted. The rows "block min" and "block max" show how much one block of 40
showers can differ from another, and "range / rms" gives their difference as a fraction of the
5-block rms: mostly between 2 and 20 per cent, and for the shape dictionary about half at three
crossings and a third at four to five, where a few clusters with large errors fall into one block
or another. Our test block is one of these five, and the numbers of this notebook should be read
with that spread in mind.

### 7c. Six test reticles, from sparse to dense

The scores are averages over many reticles. The figure shows single test reticles at six
densities: about 1, 3, 5, 10, 20 and 45 crossings. For each density we take the test reticles
whose truth rounds to it (or to the nearest value that test reticles have) and, of these, the one
in the middle when they are ordered by the error of gbt. So each row shows a typical case, not the
best or the worst. Left: the charge, with a white dot on the brightest pixel of each counted
crossing (as in notebook 10). Middle: the hit map, with the truth and every estimate above it.
Right: the clusters, coloured.

In [ ]:
tb_t = np.rint(TRUTH_T)
filled = np.unique(tb_t[tb_t > 0])
SIX = []
for t in (1, 3, 5, 10, 20, 45):
    v = filled[np.argmin(np.abs(filled - t))]           # t itself, or the nearest value that test reticles have
    c = np.flatnonzero(tb_t == v)
    c = c[np.argsort(P['gbt'][c] - TRUTH_T[c], kind='stable')]
    SIX.append((t, int(c[len(c) // 2])))                # the middle one, ordered by the error of gbt
counted = R1['part_q_total'] > THRESHOLD
fig, axs = plt.subplots(len(SIX), 3, figsize=(13, 2.05 * len(SIX)), constrained_layout=True)
for (t, j), (a0, a1, a2) in zip(SIX, axs):
    i = TEST_IDX[j]
    q = img[i]
    im = a0.imshow(np.where(q >= NOISE, q, np.nan), origin='lower', interpolation='nearest', extent=EXT,
                   norm=LogNorm(vmin=NOISE, vmax=1e5), cmap='viridis')
    rr = np.flatnonzero((R1['part_reticle'] == i) & counted)
    a0.plot((R1['part_iv'][rr] + 0.5) * PX, (R1['part_iu'][rr] + 0.5) * PY, 'o', ms=3.5, mfc='white', mec='k', mew=0.5)
    a0.set_title('charge; white dots: the brightest pixels of the crossings')
    draw_hits(a1, HITS1[i], f"about {t}: truth {TRUTH_T[j]:.2f}\n" + ', '.join(f"{nm} {P[nm][j]:.1f}" for nm in NAMES))
    lab, _ = cluster_label_image(D1, i)
    n_c = D1['n_clusters'][i]
    draw_labels(a2, lab, f"{D1['n_hits'][i]} hit pixels, {n_c} cluster{'' if n_c == 1 else 's'}")
    for ax in (a0, a1, a2):
        ax.set_xlabel('v [um]', fontsize=8)
        ax.set_ylabel('u [um]', fontsize=8)
        ax.tick_params(labelsize=8)
        ax.title.set_fontsize(9)
fig.colorbar(im, ax=axs[:, 0], label='charge per pixel [e]', location='left', shrink=0.4)
plt.show()

Up to five crossings the clusters are separate, about one per crossing, and the cluster count, the
shape dictionary and the trees come within about a crossing of the truth; the hit count falls short
at 3 and 5. From ten crossings up there are fewer clusters than crossings, the clusters grow by
merging, and the estimates spread apart: at about 45 crossings one cluster covers much of the
reticle and the cluster count gives less than half of the truth.

## 8. Where it goes wrong

A gallery of test reticles: the largest error of the trees, the largest error of the cluster
count and of the hit count among reticles with at most five crossings, and two dense reticles
picked at random near 20 and 45 crossings. Left: the charge, with a white dot on the brightest pixel of
each counted crossing (as in notebook 10). Right: the hit map with its clusters coloured, and
every estimate against the truth in the title.

In [ ]:
err = {nm: P[nm] - TRUTH_T for nm in NAMES}
few = TRUTH_T <= 5
gal = [('largest error of gbt', int(np.argmax(np.abs(err['gbt'])))),
       ('largest error of clusters, at most 5 crossings', int(np.flatnonzero(few)[np.argmax(np.abs(err['clusters'][few]))])),
       ('largest error of hits, at most 5 crossings', int(np.flatnonzero(few)[np.argmax(np.abs(err['hits'][few]))]))]
rng = np.random.default_rng(SEED + 3)
for t in (20, 45):
    c = np.flatnonzero(np.abs(TRUTH_T - t) < 0.06 * t)
    gal.append((f'a reticle with about {t} crossings', int(rng.choice(c))))
counted = R1['part_q_total'] > THRESHOLD
fig, axs = plt.subplots(len(gal), 2, figsize=(12, 2.6 * len(gal)), constrained_layout=True)
for (why, j), (a0, a1) in zip(gal, axs):
    i = TEST_IDX[j]
    q = img[i]
    im = a0.imshow(np.where(q >= NOISE, q, np.nan), origin='lower', interpolation='nearest', extent=EXT,
                   norm=LogNorm(vmin=NOISE, vmax=1e5), cmap='viridis')
    rr = np.flatnonzero((R1['part_reticle'] == i) & counted)
    a0.plot((R1['part_iv'][rr] + 0.5) * PX, (R1['part_iu'][rr] + 0.5) * PY, 'o', ms=3.5, mfc='white', mec='k', mew=0.5)
    a0.set_title(f"{why}\nn_frac = {TRUTH_T[j]:.2f}, layer {D1['layer'][i]}, {D1['r_mm'][i]:.2f} mm from the axis",
                 fontsize=9)
    lab, _ = cluster_label_image(D1, i)
    est_txt = ', '.join(f"{nm} {P[nm][j]:.1f}" for nm in NAMES)
    draw_labels(a1, lab, f"{D1['n_hits'][i]} hit pixels, {D1['n_clusters'][i]} clusters\n{est_txt}")
    a1.title.set_fontsize(9)
    for ax in (a0, a1):
        ax.set_xlabel('v [um]', fontsize=8)
        ax.set_ylabel('u [um]', fontsize=8)
fig.colorbar(im, ax=axs[:, 0], label='charge per pixel [e]', location='left', shrink=0.4)
plt.show()
tb_test = np.rint(TRUTH_T)
h_test = D1['n_hits'][TEST_IDX]
for why, j in gal:
    i = TEST_IDX[j]
    rr = np.flatnonzero((R1['part_reticle'] == i) & counted)
    ci = np.flatnonzero(cl['reticle'] == i)
    lo, hi = next((lo, hi) for lo, hi in mc.TRUTH_BINS if tb_test[j] >= lo and (hi is None or tb_test[j] <= hi))
    in_bin = (tb_test >= lo) & ((tb_test <= hi) if hi is not None else True)
    print(f"{why}: truth {TRUTH_T[j]:.2f}; " + ', '.join(f"{nm} {P[nm][j]:.2f}" for nm in NAMES))
    n_half = int(np.sum(W_ROW[rr] < 0.5))
    print(f"    {len(rr)} dots, of which {n_half} {'holds' if n_half == 1 else 'hold'} less than half of "
          f"{'its' if n_half == 1 else 'their'} crossing's charge; "
          f"{D1['n_hits'][i]} hit pixels in {len(ci)} cluster{'' if len(ci) == 1 else 's'}, "
          f"{int(cl['edge'][ci].sum())} touching the border, "
          f"{int(np.sum(cl['t_n_frac'][ci] == 0))} without a brightest pixel; largest cluster {cl['size'][ci].max()} pixels")
    print(f"    {D1['n_hits'][i] / TRUTH_T[j]:.1f} hit pixels per crossing, against "
          f"{h_test[in_bin].sum() / TRUTH_T[in_bin].sum():.1f} for all test reticles of the bin {mc._bins_label(lo, hi)}")

The first three rows are the worst cases. In all three the crossings light two to five times
more pixels than is usual for their count (the printed hit pixels per crossing, against the mean
of the bin).

- In the first and third rows most of these pixels join into one large cluster; in the third it
  is made of long slanted lines, many columns long. The hit count, the shape dictionary and the
  trees read such a map as many crossings. The cluster count sees a few clusters and gives a few
  crossings: too few in both rows.
- The second row fails for the cluster count: it shows more clusters than crossings. Two clusters
  hold no brightest pixel (the long line along u and the small piece below it: parts of a crossing
  whose brightest pixel lies in another cluster), and several at the border hold small shares of crossings that lie mostly in
  the neighbouring reticles (the printed dots with less than half of their charge). The cluster
  table gives this number of clusters the mean count of the training reticles with as many
  clusters, which are much denser.

The last two rows are dense reticles close to the axis (the distance is in the titles), with fewer
clusters than crossings. The shape dictionary falls far short in both, and the trees and the hit
count come closest.

## 9. One layer at a time

A calorimeter adds up the counts of a whole sensor plane, layer by layer. Here we add the
estimates of all reticles of one plane on the module that the shower axis crosses, and compare
with the true crossings of that plane. First one test shower, layer by layer.

In [ ]:
sh_t, lay_t, mod_t = D1['super_evt'][TEST] - N_TRAIN, VT['layer'], D1['module'][TEST]
n_test = int(R1['n_events']) - N_TRAIN
ax_t = mod_t == AXIS_MODULE
cell = sh_t * N_LAYERS + lay_t
TRUE_PL = np.bincount(cell[ax_t], TRUTH_T[ax_t], n_test * N_LAYERS).reshape(n_test, N_LAYERS)
EST_PL = {nm: np.bincount(cell[ax_t], P[nm][ax_t], n_test * N_LAYERS).reshape(n_test, N_LAYERS) for nm in NAMES}
off_share = TRUTH_T[~ax_t].sum() / TRUTH_T.sum()
tot_sh = TRUE_PL.sum(1)
s0 = int(np.argmin(np.abs(tot_sh - np.median(tot_sh))))
L = np.arange(N_LAYERS)
fig, axs = plt.subplots(2, 1, figsize=(11, 6.4), sharex=True, gridspec_kw={'height_ratios': [2, 1]},
                        constrained_layout=True)
axs[0].plot(L, TRUE_PL[s0], 'o', color='k', ms=5, label='true crossings')
for nm in NAMES:
    axs[0].plot(L, EST_PL[nm][s0], STYLE[nm], color=COLOUR[nm], label=nm)
    with np.errstate(invalid='ignore', divide='ignore'):
        axs[1].plot(L, EST_PL[nm][s0] / TRUE_PL[s0], STYLE[nm], color=COLOUR[nm])
axs[1].axhline(1, color='0.5', lw=0.8)
axs[0].set_ylabel('crossings on the plane')
axs[0].set_title(f'test shower {N_TRAIN + s0}, module {AXIS_MODULE} (the axis module)')
axs[0].legend(fontsize=9)
axs[1].set_ylabel('estimate / truth')
axs[1].set_ylim(0.4, 1.6)
axs[1].set_xlabel('layer')
plt.show()
print(f"shower {N_TRAIN + s0}: {tot_sh[s0]:,.0f} true crossings on the axis module; summed estimates: "
      + ', '.join(f"{nm} {EST_PL[nm][s0].sum():,.0f}" for nm in NAMES))
print(f"(the other modules hold {off_share:.2%} of the test crossings)")

The estimates follow the rise and fall of the shower. The ratio shows the pattern that the next
figure confirms over all 40 test showers: in the first layers the tables count too few, and in
the last layers the hit count counts too many. For the ratio over all test showers we keep, in
each layer, the planes with at least `PLANE_MIN` true crossings, and draw the mean and the
standard deviation of estimate / truth over the showers.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
for k, nm in enumerate(NAMES):
    with np.errstate(invalid='ignore', divide='ignore'):
        r = np.where(TRUE_PL >= PLANE_MIN, EST_PL[nm] / TRUE_PL, np.nan)
    n_ok = np.sum(np.isfinite(r), axis=0)
    good = n_ok >= 5
    mu = np.array([np.nanmean(r[:, l]) if good[l] else np.nan for l in L])
    sd = np.array([np.nanstd(r[:, l]) if good[l] else np.nan for l in L])
    x = L + 0.08 * (k - 2)
    ax.errorbar(x, mu, sd, fmt='o', ms=3, color=COLOUR[nm], capsize=0, lw=1, label=nm, mfc='white' if nm == 'gbt_layer' else COLOUR[nm])
ax.axhline(1, color='0.5', lw=0.8)
ax.set_xlabel('layer')
ax.set_ylabel('estimate / truth on the plane')
ax.set_title(f'test showers, axis module: mean and standard deviation over the showers (planes with at least {PLANE_MIN} crossings)',
             fontsize=10)
ax.legend(fontsize=9, ncol=5, loc='lower center')
ax.set_ylim(0.3, 1.5)
plt.show()
n_planes = np.sum(TRUE_PL >= PLANE_MIN, axis=0)
print("test showers with a plane of at least", PLANE_MIN, "crossings, per layer:", ' '.join(str(x) for x in n_planes))

The write-up (section 6.4) reports the same ratio in groups of layers, summed over all reticles of
all modules. Below are our values for the 40 test showers, with their standard deviation over
resamplings of the 40 showers (a bootstrap). The write-up's values are over all 200 showers, each
estimated in its block, so the comparison uses the five-block estimates of section 7b: the tables
must give the write-up's numbers to its printed digits; the trees are printed, as in 7b.

The two rows above the table, the share of the crossings in each group and the hit pixels per
crossing, need no training and are computed on all 200 showers; they must equal the write-up's.

In [ ]:
W64 = WRITEUP['section_6_4']
G = W64['groups']
grp = np.zeros(N_LAYERS, int)
for g, (a, b) in enumerate(G):
    grp[a:b + 1] = g
nG = len(G)
g_all = grp[D1['layer']]
share = np.bincount(g_all, D1['n_frac'], nG) / D1['n_frac'].sum()
hpc = np.bincount(g_all, D1['n_hits'], nG) / np.bincount(g_all, D1['n_frac'], nG)
assert np.allclose(share, W64['share'], atol=0.0005 + 1e-9) and np.allclose(hpc, W64['hits_per_crossing'], atol=0.05 + 1e-9), \
    "the shares or hit pixels per crossing differ from the write-up"
print("layers                 " + ''.join(f"{f'{a} to {b}':>16s}" for a, b in G))
print("share of the crossings " + ''.join(f"{x:16.1%}" for x in share))
print("hit pixels per crossing" + ''.join(f"{x:16.2f}" for x in hpc))
print("check: both rows equal the write-up's to its printed digits (all 200 showers)")

g_t = grp[lay_t]
boot = np.random.default_rng(SEED + 4).multinomial(n_test, np.full(n_test, 1 / n_test), size=2000)
t_sg = np.bincount(sh_t * nG + g_t, TRUTH_T, n_test * nG).reshape(n_test, nG)
GROUP_RATIO, DEV5 = {}, {}
print("\nestimate / truth, test showers (+- bootstrap standard deviation), all 200 showers in 5 blocks, and the write-up")
for nm in NAMES:
    e_sg = np.bincount(sh_t * nG + g_t, P[nm], n_test * nG).reshape(n_test, nG)
    r = e_sg.sum(0) / t_sg.sum(0)
    sig = ((boot @ e_sg) / (boot @ t_sg)).std(0)
    GROUP_RATIO[nm] = r
    line = ''.join(f"{x:10.3f} +-{s:.3f}" for x, s in zip(r, sig))
    print(f"{nm:14s}{line}")
    if nm in W64['ratio']:
        doc = np.array(W64['ratio'][nm])
        r5 = np.bincount(g_all, PB[nm], nG) / np.bincount(g_all, D1['n_frac'], nG)
        print(f"{'  5 blocks':14s}" + ''.join(f"{x:16.3f}" for x in r5))
        print(f"{'  write-up':14s}" + ''.join(f"{x:16.2f}" for x in doc))
        DEV5[nm] = np.abs(r5 - doc).max()
for nm in ('hits', 'clusters'):
    assert DEV5[nm] <= 0.006, f"{nm}: a five-block layer-group ratio differs from the write-up by {DEV5[nm]:.3f}"
    print(f"check: {nm}: the five-block ratios equal the write-up's to its printed digits (largest difference "
          f"{DEV5[nm]:.3f})")
for nm in ('gbt', 'gbt_layer'):
    print(f"{nm}: the five-block ratios differ from the write-up's by up to {DEV5[nm]:.3f}; printed, not asserted")

The hit-count and cluster-count tables are well below one in the first five layers; the hit count
is above one in the last ten. The trees are closer, and with the layer number the ratio is flat.
Why the front layers? The printout below compares the groups of layers on all 200 showers (no
training involved): how many of the crossings sit in dense reticles, how many pixels a crossing
lights at a given density, and the momentum of the particles (the median over the counted
crossings, taken from the particle rows of the file). The last two columns are the hit-count
table's estimate / truth in sparse and in dense test reticles.

In [ ]:
nf = D1['n_frac']
kb = np.rint(nf)
own = W1['n_own'] > 0                                    # one row per counted crossing: the reticle holding most of it
row_layer = D1['layer'][R1['part_reticle']]
row_k = kb[R1['part_reticle']]
p_all = R1['part_p_MeV']
FRONT = {}
print("layers    crossings in      hit pixels per crossing         median momentum [MeV/c]     hits estimate / truth")
print("          reticles >= 6    reticles of 1   of 6 to 20       all     in reticles >= 6     <= 5      >= 6")
for g, (a, b) in enumerate(G):
    in_g = g_all == g
    dense, single, mid = in_g & (kb >= 6), in_g & (kb == 1), in_g & (kb >= 6) & (kb <= 20)
    row_g = own & (grp[row_layer] == g)
    tg = g_t == g
    sp_, de_ = tg & (np.rint(TRUTH_T) <= 5), tg & (np.rint(TRUTH_T) >= 6)
    FRONT[g] = dict(dense=nf[dense].sum() / nf[in_g].sum(), h1=D1['n_hits'][single].sum() / nf[single].sum(),
                    hmid=D1['n_hits'][mid].sum() / nf[mid].sum(), p=np.median(p_all[row_g]),
                    pd=np.median(p_all[row_g & (row_k >= 6)]),
                    rs=P['hits'][sp_].sum() / TRUTH_T[sp_].sum(), rd=P['hits'][de_].sum() / TRUTH_T[de_].sum())
    f_ = FRONT[g]
    print(f"{a:2d} to {b:2d}  {f_['dense']:12.1%}  {f_['h1']:14.2f}  {f_['hmid']:10.2f}  {f_['p']:12.1f}  {f_['pd']:15.1f}"
          f"  {f_['rs']:12.2f}  {f_['rd']:8.2f}")

Three things set the first five layers apart. About half of their crossings sit in reticles with
six or more crossings, against a few per cent in the last ten layers. The particles in those dense
reticles have two and a half to four and a half times the momentum of those in the dense reticles
further back. And at the same density (6 to 20 crossings) a crossing there lights fewer pixels.
Notebook 10 (sections 3 and 7) saw the same near the shower axis: particles of high momentum cross
the silicon nearly at right angles and light fewer pixels each.

A table built on all layers maps a hit count to the average number of crossings behind it over all
layers. In the dense front reticles a hit count stands for more crossings than that average, so
the table gives too few there (last column), while in the sparse reticles of every group it is
close to right or a little high (the column before). In the last ten layers almost all crossings
sit in sparse reticles, where the table runs highest, and the hit count ends above one. The
trees, which see more of the map, are the closest in four of the five groups (at layers 10 to 14
the shape dictionary is closer: 0.997 against 1.010); with the layer number as an input the ratio
is flat.

## 10. Higher density: stacks of eight showers

`ret_K8.npz` stacks 8 showers plane by plane. We take the stacks made only of test showers and
apply the estimators trained on single training showers (sections 3 to 6). **This is an
extrapolation**: the training sample has few reticles this dense, and only a handful above 80
crossings (printed below). It is
not the training of the write-up's section 6.3, which trains on a mix of single showers and
stacks of 2, 4 and 8, nor of its extrapolation test in section 6.4, which trains on stacks of up
to 2.

First we read from the file which showers each stack holds.

In [ ]:
row_stack = R8['super_evt'][R8['part_reticle']]
STACK_SHOWERS = [np.unique(R8['part_evt'][row_stack == s]) for s in range(D8['n_super_events'])]
test_stacks = [s for s, ev in enumerate(STACK_SHOWERS) if ev.min() >= N_TRAIN]
mixed = [s for s, ev in enumerate(STACK_SHOWERS) if ev.min() < N_TRAIN <= ev.max()]
assert not mixed, f"stacks {mixed} hold training and test showers"
in_test = np.concatenate([STACK_SHOWERS[s] for s in test_stacks])
assert np.array_equal(np.sort(in_test), np.arange(N_TRAIN, R1['n_events'])), "the test stacks are not the test showers"
for s in test_stacks:
    print(f"stack {s}: showers {', '.join(str(x) for x in STACK_SHOWERS[s])}")
print(f"check: the {len(test_stacks)} stacks above hold exactly the test showers, each once; no stack mixes "
      f"training and test showers")
TEST8 = np.isin(D8['super_evt'], test_stacks)
V8 = mc.view([D8], [TEST8])
TRUTH8 = V8['n_frac']
P8, SC8 = {}, {}
for e in EST.values():
    P8.update({nm: v for nm, v in e.predict(V8).items() if nm in NAMES})
for nm in NAMES:
    SC8[nm] = score(P8[nm], D8, TEST8)
print(f"\n{int(TEST8.sum()):,} reticles of the test stacks, {TRUTH8.sum():,.0f} crossings, the densest {TRUTH8.max():.0f}")
print(f"training reticles (single showers) in the bin 81+: {int(np.sum(np.rint(D1['n_frac'][fit]) >= 81))}, "
      f"the densest holds {D1['n_frac'][fit].max():.1f}; test-stack reticles in that bin: "
      f"{int(np.sum(np.rint(TRUTH8) >= 81))}")
show(SC8, list(NAMES))
STACK_RATIO = {}
for nm in NAMES:
    st8 = D8['super_evt'][TEST8]
    r = np.array([P8[nm][st8 == s].sum() / TRUTH8[st8 == s].sum() for s in test_stacks])
    STACK_RATIO[nm] = r
print("\nestimate / truth summed over each test stack:")
for nm in NAMES:
    print(f"  {nm:14s} " + ' '.join(f"{x:.3f}" for x in STACK_RATIO[nm]) + f"   all stacks {SC8[nm]['total']:.3f}")
print("\nat the same true count, single test showers (K1) against test stacks (K8):")
print("true count              " + ''.join(f"{mc._bins_label(lo, hi):>8s}" for lo, hi in mc.TRUTH_BINS[4:10]))
for tag, nf_, nh_, nc_ in (('K1', TRUTH_T, VT['n_hits'], VT['n_clusters']), ('K8', TRUTH8, V8['n_hits'], V8['n_clusters'])):
    kk = np.rint(nf_)
    sel_ = [(kk >= lo) & (kk <= hi) for lo, hi in mc.TRUTH_BINS[4:10]]
    print(f"{tag} hit pixels per crossing" + ''.join(f"{nh_[s].sum() / nf_[s].sum():8.2f}" for s in sel_))
    print(f"{tag} clusters per crossing  " + ''.join(f"{nc_[s].sum() / nf_[s].sum():8.2f}" for s in sel_))

The figure shows the mean estimate divided by the mean truth in each bin, for the test stacks
(thick lines) and for the single test showers (thin). Where the two differ at the same true count, the
calibration learned on single showers does not carry over.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
for nm in NAMES:
    for sc_, lw, alpha, lab in ((SC, 1, 0.5, None), (SC8, 2, 1, nm)):
        rows_ = [r for r in sc_[nm]['bins'] if r['N'] and r['truth'] >= 0.5]
        ax.plot([r['truth'] for r in rows_], [r['est'] / r['truth'] for r in rows_], STYLE[nm], color=COLOUR[nm],
                lw=lw, alpha=alpha, label=lab)
ax.axhline(1, color='0.5', lw=0.8)
ax.set_xscale('log')
ax.set_xlabel('mean true count of the bin (n_frac)')
ax.set_ylabel('mean estimate / mean truth')
ax.set_title('trained on single showers; thick: test stacks of 8, thin: single test showers', fontsize=10)
ax.legend(fontsize=9)
plt.show()

At the same true count, a reticle of the stacks shows more hit pixels per crossing than a reticle
of a single shower in every bin of the last printed table. Its clusters per crossing are a little
higher up to 8 crossings and a little lower from 9 up. Notebook 10 (section 7) found why for the
hit pixels: the stacks reach a given density farther from the shower axis, where crossings light
more pixels each.

- Trained on single showers, the hit count, the shape dictionary and the trees read such a map as
  denser than it is: in the middle bins their bias is well above their bias on single showers
  (section 7).
- The cluster count, whose clusters per crossing change little, shifts much less; from 9 to 20
  crossings it even has a smaller rms than the trees, and it collapses in the densest bins as on
  single showers.
- Above 80 crossings, where the training holds only a handful of reticles, every binary estimator
  is far too low.
- The sums over each stack, over all its layers and modules, show what is left of this in a
  total: the trees are closest, the hit count far too high and the cluster count far too low.

A calibration for a denser shower core needs training samples of that density (section 6.4 of the
write-up reaches the same conclusion with its own training).

## 11. What the charge adds

All four estimators see one bit per pixel. As a reference, the study code has one estimator that
sees the charge: `analog`, a table of the mean count against the total charge of the reticle
(`q_sum`, before noise), in bins of equal training population. The edge logic would not have
this number. Here it is next to the binary estimators in the dense bins, for single test showers
and for the test stacks.

In [ ]:
EST['analog'] = mc.Estimators(['analog']).fit(V_TRAIN, 'n_frac')
P['analog'] = EST['analog'].predict(VT)['analog']
P8['analog'] = EST['analog'].predict(V8)['analog']
SC['analog'] = score(P['analog'])
SC8['analog'] = score(P8['analog'], D8, TEST8)
for tag, sc_ in (('single test showers', SC), ('test stacks of 8', SC8)):
    print(tag)
    show(sc_, list(NAMES) + ['analog'], what=('rms',))
    print()
for tag, sc_ in (('single test showers', SC), ('test stacks', SC8)):
    win = [r['bin'] for b, r in enumerate(sc_['analog']['bins'])
           if r['N'] and min(list(NAMES) + ['analog'], key=lambda nm: sc_[nm]['bins'][b]['rms']) == 'analog']
    print(f"{tag}: the charge sum has the smallest rms of all in the bins {', '.join(win)}")
one_c = fit & (np.abs(D1['n_frac'] - 1) < 0.02)
q10, q90 = np.percentile(D1['q_sum'][one_c], [10, 90])
print(f"charge of training reticles holding one whole crossing: {q10:,.0f} to {q90:,.0f} e (10th to 90th percentile)")
few = np.rint(TRUTH_T) <= 5
jw = int(np.flatnonzero(few)[np.argmax(np.abs(P['analog'][few] - TRUTH_T[few]))])
print(f"its largest error at 5 crossings or fewer: truth {TRUTH_T[jw]:.2f}, charge {VT['q_sum'][jw]:,.0f} e, "
      f"estimate {P['analog'][jw]:.1f}")

The charge sum is about as good as the hit count in the sparse bins, because the charge of a
single crossing varies over a wide range (printed above), and a few reticles with a very large
charge for their count spoil single bins (the printed worst case). From 13 crossings up it has the
smallest rms of all, in single showers and in the stacks, and above 80 crossings by a wide
margin. The charge of a reticle is the sum of the charges of its crossings, merged or not, while
the hit pixels and the clusters per crossing fall as the density rises. Trained on single
showers, it too is off in the summed ratio of the stacks. Section 6.3 of the write-up shows that
two bits of charge per pixel, read by a small convolutional network, recover most of what the
full charge adds over the binary map. No network is trained here.

## 12. Summary

The points below are printed from the numbers measured in this notebook.

In [ ]:
def rms_at(sc_, nm, b):
    return next(r['rms'] for r in sc_[nm]['bins'] if r['bin'] == b)


def bias_at(sc_, nm, b):
    return next(r['bias'] for r in sc_[nm]['bins'] if r['bin'] == b)


print(f"1. Data: {D1['n']:,} reticles of single 50 GeV showers; tables and models fitted on showers 0 to "
      f"{N_TRAIN - 1} ({int(TRAIN.sum()):,} reticles), scored on showers {N_TRAIN} to {R1['n_events'] - 1} "
      f"({int(TEST.sum()):,} reticles); the split is checked from the file.")
print(f"2. Hit count: rms {rms_at(SC, 'hits', '1'):.2f} at one crossing and {rms_at(SC, 'hits', '21-40'):.2f} at 21 to 40; "
      f"binned by the truth its estimates are pulled toward the typical reticle (bias "
      f"{bias_at(SC, 'hits', '0'):+.2f} in the bin '0', {bias_at(SC, 'hits', '41-80'):+.2f} at 41 to 80).")
print(f"3. Cluster count: the best simple rule in sparse reticles (rms {rms_at(SC, 'clusters', '1'):.2f} at one crossing, "
      f"better than the hit count in the bins {', '.join(better)}); clusters per crossing fall from "
      f"{CL_PER_CROSSING['1']:.2f} at one crossing to {CL_PER_CROSSING['41-80']:.2f} at 41 to 80, and the "
      f"bias there is {bias_at(SC, 'clusters', '41-80'):+.1f}.")
print(f"4. Shape dictionary: {len(KEYS):,} shapes; the 10 most common hold {KCNT[TOP].sum() / n_cl_train:.0%} of the "
      f"clusters, with mean contents from {KMEAN[TOP].min():.2f} to {KMEAN[TOP].max():.2f} crossings; rms "
      f"{rms_at(SC, 'cluster_shape', '6-8'):.2f} at 6 to 8 against {rms_at(SC, 'clusters', '6-8'):.2f} for the cluster "
      f"count, but {rms_at(SC, 'cluster_shape', '41-80'):.2f} at 41 to 80.")
print(f"5. Boosted trees on 49 numbers: the smallest rms of the four in {len(GBT_BEST)} of {N_BINS_T} bins; rms "
      f"{rms_at(SC, 'gbt', '1'):.2f} at one crossing, {rms_at(SC, 'gbt', '13-20'):.2f} at 13 to 20, "
      f"{rms_at(SC, 'gbt', '41-80'):.2f} at 41 to 80. Summed over a sensor plane they scatter by "
      f"{SC['gbt']['plane_std']:.1%} per plane, the tables by {min(SC[nm]['plane_std'] for nm in FOUR[:3]):.1%} "
      f"to {max(SC[nm]['plane_std'] for nm in FOUR[:3]):.1%}.")
print(f"6. Refitted over the write-up's five blocks, the tables reproduce section 6.1 to its printed digits and the "
      f"trees within {TOL['gbt'].max():.1%}.")
print(f"7. Layers 0 to 4: estimate / truth {GROUP_RATIO['hits'][0]:.2f} (hits), {GROUP_RATIO['clusters'][0]:.2f} "
      f"(clusters), {GROUP_RATIO['gbt'][0]:.2f} (gbt), {GROUP_RATIO['gbt_layer'][0]:.2f} (gbt_layer). Early crossings "
      f"sit in denser reticles ({FRONT[0]['dense']:.0%} in reticles of 6 or more) and light fewer pixels "
      f"({FRONT[0]['hmid']:.2f} against {FRONT[2]['hmid']:.2f} per crossing at 6 to 20, layers 0 to 4 against "
      f"10 to 14); with the layer as input the ratio is flat.")
print(f"8. Stacks of 8 test showers, estimators trained on single showers (an extrapolation): estimate / truth "
      f"summed over the stacks " + ', '.join(f"{nm} {SC8[nm]['total']:.3f}" for nm in NAMES) + ".")
print(f"9. The charge sum (not available to a binary readout): rms {rms_at(SC, 'analog', '21-40'):.2f} at 21 to 40 against "
      f"{rms_at(SC, 'gbt', '21-40'):.2f} for the trees; in the stacks above 80 crossings {rms_at(SC8, 'analog', '81+'):.1f} "
      f"against {min(rms_at(SC8, nm, '81+') for nm in NAMES):.1f} for the best binary estimator.")
print(f"10. A library of every exact shape (section 5b): {len(E_KEYS):,} shapes over all 200 showers, and "
      f"{np.mean(e_cnt[e_inv[large]] == 1):.1%} of the clusters of 17 pixels or more have a shape seen once; rms "
      f"{rms_at(SC, 'exact-shape library', '13-20'):.2f} at 13 to 20 against {rms_at(SC, 'cluster_shape', '13-20'):.2f} "
      f"for the shape dictionary and {rms_at(SC, 'gbt', '13-20'):.2f} for the trees.")
print("\nNext, notebook 12: from the reticle counts to a layer profile and a shower total.")
peak = None
try:
    import resource
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1e6 if sys.platform == 'darwin' else 1e3)
except ImportError:
    pass
print(f"\nrun time {time.time() - T_START:.0f} s" + (f", peak memory {peak:,.0f} MB" if peak else ""))